# GenoHybrid-VEP — Notebook 2
## Tri-Modal Model Training

This notebook loads the cleaned feature table produced by Notebook 1 and begins the final GenoHybrid-VEP model pipeline.

## Imports, reproducibility and device

In [2]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


## Mount Google Drive and load Notebook 1 output

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/"
    "variant_effect_prediction"
)

DATA_PATH = (
    PROJECT_PATH /
    "data" /
    "processed" /
    "genohybrid_features.parquet"
)

CHECKPOINT_DIR = (
    PROJECT_PATH /
    "checkpoints"
)

RESULTS_DIR = (
    PROJECT_PATH /
    "results"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


df = pd.read_parquet(
    DATA_PATH
)


print(
    "Dataset shape:",
    df.shape
)

print(
    "\nColumns:"
)

print(
    df.columns.tolist()
)

Dataset shape: (100000, 36)

Columns:
['node_id', 'variant_key', 'Chromosome', 'Start', 'Stop', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'GeneSymbol', 'Type', 'ClinicalSignificance', 'Label', 'Assembly', 'OriginSimple', 'ReviewStatus', 'NumberSubmitters', 'SomaticClinicalImpact', 'Ref_A', 'Ref_C', 'Ref_G', 'Ref_T', 'Alt_A', 'Alt_C', 'Alt_G', 'Alt_T', 'Tab_Scaled_Submitters', 'Tab_Origin_Numeric', 'Tab_Review_Score', 'Tab_Somatic_Impact', 'seq201', 'phylop', 'phastcons', 'molecular_consequence_raw', 'consequence_category', 'gnomad_af', 'gnomad_present', 'ref_match']


## Dataset audit

This cell checks that the output from Notebook 1 is valid before any modeling starts.

Expected:
- `seq201` should always have length 201
- every `ref_match` should be `True`
- `node_id` should be unique
- labels should only be 0, 1, 2
- gnomAD may still be completely missing at this stage

In [5]:
print(
    "================ DATASET AUDIT ================"
)

print(
    "\nVariants:",
    f"{len(df):,}"
)


print(
    "\nClass counts:"
)

print(
    df["Label"]
    .value_counts()
    .sort_index()
)


print(
    "\nClass percentages:"
)

print(
    (
        df["Label"]
        .value_counts(
            normalize=True
        )
        .sort_index()
        * 100
    ).round(2)
)


print(
    "\nSequence lengths:"
)

print(
    df["seq201"]
    .str.len()
    .value_counts()
    .head()
)


print(
    "\nREF match:"
)

print(
    df["ref_match"]
    .value_counts(
        dropna=False
    )
)


print(
    "\nMissing biological features:"
)

for column in [
    "phylop",
    "phastcons",
    "consequence_category",
    "gnomad_af"
]:

    if column in df.columns:
        missing = (
            df[column]
            .isna()
            .mean()
        )

        print(
            f"{column:25s}: "
            f"{missing:.2%}"
        )

    else:
        print(
            f"{column:25s}: "
            "COLUMN MISSING"
        )


# ============================================================
# HARD CHECKS
# ============================================================

assert (
    df["seq201"]
    .str.len()
    .eq(201)
    .all()
), "Some seq201 rows do not have length 201."

assert (
    df["ref_match"]
    .all()
), "Some REF alleles do not match GRCh38."

assert (
    df["node_id"]
    .is_unique
), "node_id is not unique."

assert (
    set(
        df["Label"]
        .unique()
    )
    <= {0, 1, 2}
), "Unexpected labels found."


print(
    "\n✅ Dataset passed core checks."
)

================ DATASET AUDIT ================

Variants: 100,000

Class counts:
Label
0    34621
1     5135
2    60244
Name: count, dtype: int64

Class percentages:
Label
0    34.62
1     5.14
2    60.24
Name: proportion, dtype: float64

Sequence lengths:
seq201
201    100000
Name: count, dtype: int64

REF match:
ref_match
True    100000
Name: count, dtype: int64

Missing biological features:
phylop                   : 0.00%
phastcons                : 0.00%
consequence_category     : 0.00%
gnomad_af                : 100.00%

✅ Dataset passed core checks.


## Build the gene-grouping variable

The final project evaluates on genes unseen during training.

For variants without a usable gene symbol, we create a fallback group based on:

`NO_GENE_<chromosome>_<megabase region>`

This is safer than treating every gene-less variant independently.

In [6]:
df["group_id"] = (
    df["GeneSymbol"]
    .astype("string")
)


missing_gene = (
    df["group_id"].isna()
    |
    (
        df["group_id"]
        .fillna("")
        .str.len()
        == 0
    )
)


df.loc[
    missing_gene,
    "group_id"
] = (
    "NO_GENE_"
    +
    df.loc[
        missing_gene,
        "Chromosome"
    ].astype(str)
    +
    "_"
    +
    (
        df.loc[
            missing_gene,
            "Start"
        ]
        // 1_000_000
    ).astype(str)
)


print(
    "Unique groups:",
    df["group_id"]
    .nunique()
)

print(
    "Variants without named genes:",
    int(missing_gene.sum())
)

Unique groups: 14465
Variants without named genes: 0


## Development train / validation / test split

For development, we use a single gene-separated split:

- about 80% training genes
- about 10% validation genes
- about 10% test genes

Later, after the full architecture is stable, this will be replaced with the planned 5-fold `GroupKFold`.

In [7]:
# ============================================================
# OUTER SPLIT
# train+validation vs test
# ============================================================

outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=SEED
)


trainval_idx, test_idx = next(

    outer_split.split(
        df,
        y=df["Label"],
        groups=df["group_id"]
    )
)


trainval_df = (
    df.iloc[
        trainval_idx
    ]
    .copy()
)


test_df = (
    df.iloc[
        test_idx
    ]
    .copy()
)


# ============================================================
# INNER SPLIT
# train vs validation
# ============================================================

inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1111,
    random_state=SEED
)


train_idx, val_idx = next(

    inner_split.split(
        trainval_df,
        y=trainval_df["Label"],
        groups=trainval_df["group_id"]
    )
)


train_df = (
    trainval_df.iloc[
        train_idx
    ]
    .copy()
)


val_df = (
    trainval_df.iloc[
        val_idx
    ]
    .copy()
)


print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)

Train: 80065
Validation: 10298
Test: 9637


## Prove there is no gene-group leakage

In [8]:
train_groups = set(
    train_df["group_id"]
)

val_groups = set(
    val_df["group_id"]
)

test_groups = set(
    test_df["group_id"]
)


train_val_overlap = (
    train_groups
    &
    val_groups
)

train_test_overlap = (
    train_groups
    &
    test_groups
)

val_test_overlap = (
    val_groups
    &
    test_groups
)


print(
    "Train ↔ Validation overlap:",
    len(train_val_overlap)
)

print(
    "Train ↔ Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test overlap:",
    len(val_test_overlap)
)


assert (
    len(train_val_overlap)
    == 0
)

assert (
    len(train_test_overlap)
    == 0
)

assert (
    len(val_test_overlap)
    == 0
)


print(
    "\n✅ No gene-group leakage."
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0

✅ No gene-group leakage.


## Inspect class balance in each partition

In [9]:
def describe_split(
    name,
    dataframe
):

    print(
        f"\n========== {name} =========="
    )

    print(
        dataframe["Label"]
        .value_counts()
        .sort_index()
    )

    print(
        (
            dataframe["Label"]
            .value_counts(
                normalize=True
            )
            .sort_index()
            * 100
        ).round(2)
    )


describe_split(
    "TRAIN",
    train_df
)

describe_split(
    "VALIDATION",
    val_df
)

describe_split(
    "TEST",
    test_df
)


========== TRAIN ==========
Label
0    27686
1     4110
2    48269
Name: count, dtype: int64
Label
0    34.58
1     5.13
2    60.29
Name: proportion, dtype: float64

========== VALIDATION ==========
Label
0    3500
1     549
2    6249
Name: count, dtype: int64
Label
0    33.99
1     5.33
2    60.68
Name: proportion, dtype: float64

========== TEST ==========
Label
0    3435
1     476
2    5726
Name: count, dtype: int64
Label
0    35.64
1     4.94
2    59.42
Name: proportion, dtype: float64


# Arm 1 Preparation — 101 bp DNA sequence input

Notebook 1 stores a 201 bp GRCh38 sequence centered on each variant.

For the final sequence arm we crop the central 101 bp:

- 50 bases before the variant
- the variant position
- 50 bases after the variant

The surrounding sequence is ordinary one-hot DNA.

At the centre position we use a mutation-aware encoding:

- `+1` on the reference base channel
- `-1` on the alternative base channel


## Mutation-aware sequence encoder

In [12]:
BASE_TO_INDEX = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3
}


def encode_variant_sequence(
    seq201,
    ref,
    alt,
    window_size=101
):

    assert (
        window_size % 2
        == 1
    )

    assert (
        ref in BASE_TO_INDEX
        and
        alt in BASE_TO_INDEX
    ), "REF and ALT must be A/C/G/T."

    center_201 = 100

    half = (
        window_size
        // 2
    )


    sequence = seq201[
        center_201 - half:
        center_201 + half + 1
    ]


    assert (
        len(sequence)
        == window_size
    )


    encoded = np.zeros(
        (
            4,
            window_size
        ),
        dtype=np.float32
    )


    # Ordinary one-hot DNA context
    for position, base in enumerate(
        sequence
    ):

        if base in BASE_TO_INDEX:

            encoded[
                BASE_TO_INDEX[base],
                position
            ] = 1.0


    # ========================================================
    # Mutation-aware centre encoding
    # ========================================================

    centre = half

    encoded[
        :,
        centre
    ] = 0.0


    encoded[
        BASE_TO_INDEX[ref],
        centre
    ] = 1.0


    encoded[
        BASE_TO_INDEX[alt],
        centre
    ] = -1.0


    return encoded

## Test the sequence encoding

This is the checkpoint for Part 1.

Expected:
- shape `(4, 101)`
- one positive centre channel for REF
- one negative centre channel for ALT

In [13]:
sample = train_df.iloc[0]


encoded = encode_variant_sequence(

    sample["seq201"],

    sample[
        "ReferenceAlleleVCF"
    ],

    sample[
        "AlternateAlleleVCF"
    ],

    window_size=101
)


print(
    "Encoded shape:",
    encoded.shape
)

print(
    "REF:",
    sample[
        "ReferenceAlleleVCF"
    ]
)

print(
    "ALT:",
    sample[
        "AlternateAlleleVCF"
    ]
)

print(
    "Centre encoding:",
    encoded[:, 50]
)


assert encoded.shape == (4, 101)
assert np.isclose(
    encoded[
        BASE_TO_INDEX[
            sample["ReferenceAlleleVCF"]
        ],
        50
    ],
    1.0
)
assert np.isclose(
    encoded[
        BASE_TO_INDEX[
            sample["AlternateAlleleVCF"]
        ],
        50
    ],
    -1.0
)

print(
    "\n✅ 101-bp mutation-aware sequence encoding works."
)

Encoded shape: (4, 101)
REF: G
ALT: A
Centre encoding: [-1.  0.  1.  0.]

✅ 101-bp mutation-aware sequence encoding works.


## Dataset for sequence arm

In [14]:
class SequenceDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

        self.labels = torch.tensor(
            self.df["Label"].values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = encode_variant_sequence(
            row["seq201"],
            row["ReferenceAlleleVCF"],
            row["AlternateAlleleVCF"],
            window_size=101
        )
        seq = torch.tensor(seq, dtype=torch.float32)
        label = self.labels[idx]

        return seq, label

## Create train, validation and test datasets

In [15]:
train_dataset = SequenceDataset(
    train_df
)

val_dataset = SequenceDataset(
    val_df
)

test_dataset = SequenceDataset(
    test_df
)


print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Test samples:",
    len(test_dataset)
)

Train samples: 80065
Validation samples: 10298
Test samples: 9637


## DataLoaders

In [16]:
if torch.cuda.is_available():

    BATCH_SIZE = 512

else:

    BATCH_SIZE = 256


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Batch size: 256
Train batches: 313
Validation batches: 41
Test batches: 38


## Calculate class weights

In [17]:
train_counts = (train_df["Label"].value_counts().sort_index())
num_classes = 3

num_train = len(train_df)
class_weights = []

for class_id in range(num_classes):
    count = train_counts.get(class_id, 1)
    weight = (num_train / (num_classes * count))
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:", class_weights)

Class weights: tensor([0.9640, 6.4935, 0.5529])


## 101 bp CNN arm

In [ ]:
class SequenceCNN(nn.Module):

    def __init__(
        self,
        embedding_dim=64
    ):

        super().__init__()


        # ==========================================
        # Convolution Block 1
        # ==========================================

        self.conv1 = nn.Conv1d(
            in_channels=4,
            out_channels=32,
            kernel_size=7,
            padding=3
        )

        self.bn1 = nn.BatchNorm1d(
            32
        )


        # ==========================================
        # Convolution Block 2
        # ==========================================

        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=64,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            64
        )


        # ==========================================
        # Convolution Block 3
        # ==========================================

        self.conv3 = nn.Conv1d(
            in_channels=64,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.bn3 = nn.BatchNorm1d(
            64
        )


        self.pool = nn.MaxPool1d(
            kernel_size=2
        )


        self.dropout = nn.Dropout(
            0.30
        )


        # Global pooling produces:
        #
        # average = 64
        # maximum = 64
        #
        # total = 128

        self.embedding = nn.Linear(
            128,
            embedding_dim
        )


    def forward(
        self,
        x
    ):

        # Input:
        #
        # [batch, 4, 101]


        x = self.conv1(x)

        x = self.bn1(x)

        x = F.relu(x)

        x = self.pool(x)



        x = self.conv2(x)

        x = self.bn2(x)

        x = F.relu(x)

        x = self.pool(x)



        x = self.conv3(x)

        x = self.bn3(x)

        x = F.relu(x)



        # ==========================================
        # Global pooling
        # ==========================================

        avg_pool = torch.mean(
            x,
            dim=2
        )

        max_pool = torch.max(
            x,
            dim=2
        ).values


        x = torch.cat(
            [
                avg_pool,
                max_pool
            ],
            dim=1
        )


        x = self.dropout(x)


        embedding = self.embedding(
            x
        )

        embedding = F.relu(
            embedding
        )


        return embedding

## Sequence only classifier

In [ ]:
class SequenceOnlyModel(nn.Module):

    def __init__(
        self,
        num_classes=3
    ):

        super().__init__()


        self.encoder = SequenceCNN(
            embedding_dim=64
        )


        self.classifier = nn.Sequential(

            nn.Linear(
                64,
                32
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        embedding = self.encoder(
            x
        )


        logits = self.classifier(
            embedding
        )


        return logits

## Model initialisation

In [ ]:
sequence_model = SequenceOnlyModel(
    num_classes=3
).to(device)


loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)


optimizer = torch.optim.AdamW(

    sequence_model.parameters(),

    lr=1e-3,

    weight_decay=1e-4
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=1
)


print(
    sequence_model
)


total_parameters = sum(

    p.numel()

    for p in sequence_model.parameters()

    if p.requires_grad

)


print(
    "\nTrainable parameters:",
    f"{total_parameters:,}"
)

## Initial validation

In [21]:
seq_batch, labels = next(
    iter(train_loader)
)


seq_batch = seq_batch.to(
    device
)


labels = labels.to(
    device
)


with torch.no_grad():

    logits = sequence_model(
        seq_batch
    )


print(
    "Input shape:",
    seq_batch.shape
)

print(
    "Output shape:",
    logits.shape
)

print(
    "Label shape:",
    labels.shape
)


assert (
    seq_batch.shape[1:]
    ==
    (4, 101)
)


assert (
    logits.shape[1]
    ==
    3
)


print(
    "\n✅ CNN forward pass works."
)

Input shape: torch.Size([256, 4, 101])
Output shape: torch.Size([256, 3])
Label shape: torch.Size([256])

✅ CNN forward pass works.


## Evaluation loop

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    loader
):

    model.eval()


    all_targets = []

    all_predictions = []

    all_probabilities = []


    for seq, targets in loader:

        seq = seq.to(
            device
        )

        targets = targets.to(
            device
        )


        logits = model(
            seq
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        all_targets.extend(
            targets.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


    y_true = np.asarray(
        all_targets
    )

    y_pred = np.asarray(
        all_predictions
    )

    y_prob = np.asarray(
        all_probabilities
    )


    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )


    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )


    return {

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_f1":
            macro_f1,

        "mcc":
            mcc,

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }

## Training loop

In [ ]:
EPOCHS = 15

PATIENCE = 3


best_val_f1 = -1

epochs_without_improvement = 0


best_model_path = (
    CHECKPOINT_DIR /
    "sequence_cnn_best.pt"
)


print(
    f"Training sequence CNN on {device}..."
)


for epoch in range(
    EPOCHS
):


    # ==========================================
    # TRAIN
    # ==========================================

    sequence_model.train()


    running_loss = 0.0


    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )


    for seq, targets in progress:


        seq = seq.to(
            device,
            non_blocking=True
        )


        targets = targets.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = sequence_model(
            seq
        )


        loss = loss_fn(
            logits,
            targets
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            sequence_model.parameters(),

            max_norm=5.0

        )


        optimizer.step()


        running_loss += (
            loss.item()
            *
            seq.size(0)
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    train_loss = (
        running_loss
        /
        len(train_dataset)
    )


    # ==========================================
    # VALIDATE
    # ==========================================

    val_results = evaluate_model(

        sequence_model,

        val_loader

    )


    val_f1 = (
        val_results[
            "macro_f1"
        ]
    )


    scheduler.step(
        val_f1
    )


    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    print(

        f"Epoch {epoch + 1:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "
        f"Val Balanced Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "
        f"Val Macro-F1: "
        f"{val_f1:.4f} | "
        f"Val MCC: "
        f"{val_results['mcc']:.4f} | "
        f"LR: {current_lr:.2e}"

    )


    # ==========================================
    # EARLY STOPPING
    # ==========================================

    if (
        val_f1
        >
        best_val_f1
        + 1e-4
    ):


        best_val_f1 = val_f1


        epochs_without_improvement = 0


        torch.save(

            {
                "model_state_dict":
                    sequence_model.state_dict(),

                "val_macro_f1":
                    best_val_f1,

                "epoch":
                    epoch + 1,

                "seed":
                    SEED,

                "window_size":
                    101

            },

            best_model_path

        )


        print(
            "✓ Best model saved."
        )


    else:


        epochs_without_improvement += 1


        print(

            "No improvement:",

            f"{epochs_without_improvement}/"
            f"{PATIENCE}"

        )


    if (
        epochs_without_improvement
        >=
        PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

Training sequence CNN on cpu...


Epoch 1/15:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 01 | Loss: 1.1117 | Val Acc: 0.2057 | Val Balanced Acc: 0.3537 | Val Macro-F1: 0.1978 | Val MCC: 0.0216 | LR: 1.00e-03
✓ Best model saved.


Epoch 2/15:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 02 | Loss: 1.0932 | Val Acc: 0.1340 | Val Balanced Acc: 0.3607 | Val Macro-F1: 0.1304 | Val MCC: 0.0279 | LR: 1.00e-03
No improvement: 1/3


...

Epoch 6/15:   0%|          | 0/313 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7cc06d67ab60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
Exception ignored in:   File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7cc06d67ab60>    assert self._parent_pid == os.getpid(), 'can only test a child process'
Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
AssertionError: can only test a child process    
self._shutdown_workers()Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7cc06d67ab60>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/da

Epoch 06 | Loss: 1.0066 | Val Acc: 0.4143 | Val Balanced Acc: 0.4638 | Val Macro-F1: 0.3565 | Val MCC: 0.0838 | LR: 5.00e-04
No improvement: 2/3


Epoch 7/15:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 07 | Loss: 0.9742 | Val Acc: 0.4148 | Val Balanced Acc: 0.4624 | Val Macro-F1: 0.3627 | Val MCC: 0.0944 | LR: 5.00e-04
No improvement: 3/3

Early stopping triggered.


## Evaluating against previous best model

In [22]:
checkpoint = torch.load(
    best_model_path,
    map_location=device
)


sequence_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


print(
    "Restored epoch:",
    checkpoint["epoch"]
)

train_results = evaluate_model(
    sequence_model,
    train_loader
)

print(
    "Train Macro-F1:",
    train_results["macro_f1"]
)

print(
    "Validation Macro-F1:",
    checkpoint["val_macro_f1"]
)

NameError: name 'best_model_path' is not defined

# Arm 2: MLP

## Arm 2 Features

In [26]:
# ============================================================
# ARM 2 — BIOMARKER FEATURES
# ============================================================

NUMERIC_FEATURES = [
    "phylop",
    "phastcons",
    "Tab_Review_Score",
    "Tab_Origin_Numeric",
    "Tab_Somatic_Impact"
]

CATEGORICAL_FEATURES = [
    "consequence_category"
]


print("Numeric features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

Numeric features:
['phylop', 'phastcons', 'Tab_Review_Score', 'Tab_Origin_Numeric', 'Tab_Somatic_Impact']

Categorical features:
['consequence_category']


In [27]:
# ============================================================
# FIT NUMERIC IMPUTATION ON TRAIN ONLY
# ============================================================

train_medians = {}


for feature in NUMERIC_FEATURES:

    median = (
        train_df[feature]
        .median()
    )

    # If an entire feature somehow happens to be missing,
    # use zero rather than crashing.
    if pd.isna(median):
        median = 0.0

    train_medians[
        feature
    ] = float(median)


print(
    "Training medians:"
)

print(
    train_medians
)

Training medians:
{'phylop': 1.4950000047683716, 'phastcons': 0.9909999966621399, 'Tab_Review_Score': 1.0, 'Tab_Origin_Numeric': 1.0, 'Tab_Somatic_Impact': 1.0}


In [28]:
train_consequence_categories = sorted(

    train_df[
        "consequence_category"
    ]
    .fillna("unknown")
    .astype(str)
    .unique()
)


print(
    "Training consequence categories:"
)

print(
    train_consequence_categories
)

Training consequence categories:
['intron', 'missense', 'nonsense_or_stop', 'other', 'splice', 'synonymous', 'unknown', 'utr']


## Biomarker transformation function

In [29]:
def build_biomarker_matrix(
    dataframe,
    fit_scaler=False,
    scaler=None
):

    dataframe = (
        dataframe
        .reset_index(drop=True)
        .copy()
    )


    # ========================================================
    # NUMERIC FEATURES
    # ========================================================

    numeric = dataframe[
        NUMERIC_FEATURES
    ].copy()


    # Add missing-value indicators before imputation.
    #
    # Missing conservation itself may carry useful information.
    for feature in [
        "phylop",
        "phastcons"
    ]:

        numeric[
            f"{feature}_missing"
        ] = (
            dataframe[feature]
            .isna()
            .astype(np.float32)
        )


    # Fill missing values using medians learned from TRAIN only.
    for feature in NUMERIC_FEATURES:

        numeric[feature] = (
            numeric[feature]
            .fillna(
                train_medians[feature]
            )
        )


    numeric = numeric.astype(
        np.float32
    )


    # ========================================================
    # SCALE NUMERIC FEATURES
    # ========================================================

    if fit_scaler:

        scaler = StandardScaler()

        numeric_scaled = scaler.fit_transform(
            numeric
        )

    else:

        numeric_scaled = scaler.transform(
            numeric
        )


    # ========================================================
    # CATEGORICAL: MOLECULAR CONSEQUENCE
    # ========================================================

    consequence = (
        dataframe[
            "consequence_category"
        ]
        .fillna("unknown")
        .astype(str)
    )


    consequence_features = np.zeros(

        (
            len(dataframe),
            len(
                train_consequence_categories
            )
        ),

        dtype=np.float32
    )


    category_to_index = {

        category: i

        for i, category
        in enumerate(
            train_consequence_categories
        )
    }


    for row_index, value in enumerate(
        consequence
    ):

        if value in category_to_index:

            consequence_features[
                row_index,
                category_to_index[value]
            ] = 1.0


    # ========================================================
    # FINAL MATRIX
    # ========================================================

    biomarker_matrix = np.concatenate(

        [
            numeric_scaled.astype(
                np.float32
            ),

            consequence_features
        ],

        axis=1
    )


    return (
        biomarker_matrix,
        scaler
    )

## Build train / validation / test biomarker matrices

In [30]:
X_train_tab, tab_scaler = (
    build_biomarker_matrix(
        train_df,
        fit_scaler=True
    )
)


X_val_tab, _ = (
    build_biomarker_matrix(
        val_df,
        fit_scaler=False,
        scaler=tab_scaler
    )
)


X_test_tab, _ = (
    build_biomarker_matrix(
        test_df,
        fit_scaler=False,
        scaler=tab_scaler
    )
)


print(
    "Train:",
    X_train_tab.shape
)

print(
    "Validation:",
    X_val_tab.shape
)

print(
    "Test:",
    X_test_tab.shape
)


assert (
    X_train_tab.shape[1]
    ==
    X_val_tab.shape[1]
    ==
    X_test_tab.shape[1]
)


assert np.isfinite(
    X_train_tab
).all()


assert np.isfinite(
    X_val_tab
).all()


assert np.isfinite(
    X_test_tab
).all()


print(
    "\n✅ Biomarker matrices ready."
)

Train: (80065, 15)
Validation: (10298, 15)
Test: (9637, 15)

✅ Biomarker matrices ready.


## Create biomarker datasets

In [31]:
class BiomarkerDataset(
    Dataset
):

    def __init__(
        self,
        X,
        labels
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )


        self.y = torch.tensor(
            labels,
            dtype=torch.long
        )


    def __len__(
        self
    ):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )

In [32]:
train_tab_dataset = BiomarkerDataset(

    X_train_tab,

    train_df[
        "Label"
    ].values
)


val_tab_dataset = BiomarkerDataset(

    X_val_tab,

    val_df[
        "Label"
    ].values
)


test_tab_dataset = BiomarkerDataset(

    X_test_tab,

    test_df[
        "Label"
    ].values
)


print(
    "Train:",
    len(
        train_tab_dataset
    )
)

print(
    "Validation:",
    len(
        val_tab_dataset
    )
)

print(
    "Test:",
    len(
        test_tab_dataset
    )
)

Train: 80065
Validation: 10298
Test: 9637


## Biomarker dataloader

In [33]:
TAB_BATCH_SIZE = 512


train_tab_loader = DataLoader(

    train_tab_dataset,

    batch_size=TAB_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_tab_loader = DataLoader(

    val_tab_dataset,

    batch_size=TAB_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_tab_loader = DataLoader(

    test_tab_dataset,

    batch_size=TAB_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Biomarker input dimension:",
    X_train_tab.shape[1]
)

print(
    "Train batches:",
    len(
        train_tab_loader
    )
)

Biomarker input dimension: 15
Train batches: 157


## Arm 2 MLP

In [34]:
class BiomarkerMLP(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        embedding_dim=32
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.BatchNorm1d(
                64
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),


            nn.Linear(
                64,
                32
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.20
            ),


            nn.Linear(
                32,
                embedding_dim
            ),

            nn.ReLU()
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

## Biomarker only classifier

In [35]:
class BiomarkerOnlyModel(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        num_classes=3
    ):

        super().__init__()


        self.encoder = BiomarkerMLP(

            input_dim=input_dim,

            embedding_dim=32
        )


        self.classifier = nn.Sequential(

            nn.Linear(
                32,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        embedding = self.encoder(
            x
        )


        logits = self.classifier(
            embedding
        )


        return logits

## Initialise and validate Arm 2

In [36]:
biomarker_model = BiomarkerOnlyModel(

    input_dim=
        X_train_tab.shape[1],

    num_classes=3

).to(device)


tab_loss_fn = (
    nn.CrossEntropyLoss(
        weight=class_weights
    )
)


tab_optimizer = (
    torch.optim.AdamW(

        biomarker_model.parameters(),

        lr=1e-3,

        weight_decay=1e-4
    )
)


tab_scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(

        tab_optimizer,

        mode="max",

        factor=0.5,

        patience=1
    )
)


print(
    biomarker_model
)


trainable = sum(

    p.numel()

    for p in biomarker_model.parameters()

    if p.requires_grad
)


print(
    "\nTrainable parameters:",
    f"{trainable:,}"
)

BiomarkerOnlyModel(
  (encoder): BiomarkerMLP(
    (network): Sequential(
      (0): Linear(in_features=15, out_features=64, bias=True)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=64, out_features=32, bias=True)
      (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=32, out_features=32, bias=True)
      (9): ReLU()
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)

Trainable parameters: 5,507


In [ ]:
tab_batch, labels = next(

    iter(
        train_tab_loader
    )
)


tab_batch = tab_batch.to(
    device
)


labels = labels.to(
    device
)


with torch.no_grad():

    logits = biomarker_model(
        tab_batch
    )


print(
    "Input:",
    tab_batch.shape
)

print(
    "Output:",
    logits.shape
)

print(
    "Labels:",
    labels.shape
)


assert (
    logits.shape[1]
    == 3
)


print(
    "\n✅ Biomarker MLP forward pass works."
)

Input: torch.Size([512, 15])
Output: torch.Size([512, 3])
Labels: torch.Size([512])

✅ Biomarker MLP forward pass works.


## Evaluation function

In [ ]:
@torch.no_grad()
def evaluate_tabular_model(
    model,
    loader
):

    model.eval()


    all_targets = []
    all_predictions = []
    all_probabilities = []


    for features, targets in loader:

        features = features.to(
            device
        )

        targets = targets.to(
            device
        )


        logits = model(
            features
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        all_targets.extend(
            targets.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


    y_true = np.asarray(
        all_targets
    )

    y_pred = np.asarray(
        all_predictions
    )

    y_prob = np.asarray(
        all_probabilities
    )


    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }

## Training

In [ ]:
TAB_EPOCHS = 15

TAB_PATIENCE = 3


best_tab_f1 = -1.0

tab_epochs_without_improvement = 0


best_tab_path = (

    CHECKPOINT_DIR /

    "biomarker_mlp_best.pt"
)


print(
    f"Training biomarker MLP on {device}..."
)


for epoch in range(
    TAB_EPOCHS
):

    biomarker_model.train()


    running_loss = 0.0


    progress = tqdm(

        train_tab_loader,

        desc=
            f"Epoch "
            f"{epoch + 1}/"
            f"{TAB_EPOCHS}"
    )


    for features, targets in progress:


        features = features.to(
            device,
            non_blocking=True
        )


        targets = targets.to(
            device,
            non_blocking=True
        )


        tab_optimizer.zero_grad(
            set_to_none=True
        )


        logits = biomarker_model(
            features
        )


        loss = tab_loss_fn(
            logits,
            targets
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            biomarker_model.parameters(),

            5.0
        )


        tab_optimizer.step()


        running_loss += (

            loss.item()

            *
            features.size(0)
        )


        progress.set_postfix(

            loss=
                f"{loss.item():.4f}"
        )


    train_loss = (

        running_loss

        /

        len(
            train_tab_dataset
        )
    )


    val_results = (
        evaluate_tabular_model(

            biomarker_model,

            val_tab_loader
        )
    )


    val_f1 = (
        val_results[
            "macro_f1"
        ]
    )


    tab_scheduler.step(
        val_f1
    )


    current_lr = (
        tab_optimizer
        .param_groups[0]["lr"]
    )


    print(

        f"Epoch {epoch + 1:02d} | "

        f"Loss: "
        f"{train_loss:.4f} | "

        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "

        f"Val Balanced Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "

        f"Val Macro-F1: "
        f"{val_f1:.4f} | "

        f"Val MCC: "
        f"{val_results['mcc']:.4f} | "

        f"LR: "
        f"{current_lr:.2e}"
    )


    if (
        val_f1
        >
        best_tab_f1
        + 1e-4
    ):

        best_tab_f1 = (
            val_f1
        )

        tab_epochs_without_improvement = 0


        torch.save(

            {
                "model_state_dict":
                    biomarker_model
                    .state_dict(),

                "val_macro_f1":
                    best_tab_f1,

                "epoch":
                    epoch + 1,

                "input_dim":
                    X_train_tab.shape[1],

                "numeric_features":
                    NUMERIC_FEATURES,

                "consequence_categories":
                    train_consequence_categories,

                "seed":
                    SEED
            },

            best_tab_path
        )


        print(
            "✓ Best model saved."
        )


    else:

        tab_epochs_without_improvement += 1


        print(

            "No improvement:",

            f"{tab_epochs_without_improvement}/"
            f"{TAB_PATIENCE}"
        )


    if (
        tab_epochs_without_improvement
        >=
        TAB_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

Training biomarker MLP on cpu...


Epoch 1/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.6669 | Val Acc: 0.8707 | Val Balanced Acc: 0.8281 | Val Macro-F1: 0.7776 | Val MCC: 0.7578 | LR: 1.00e-03
✓ Best model saved.


Epoch 2/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 02 | Loss: 0.4718 | Val Acc: 0.8884 | Val Balanced Acc: 0.8313 | Val Macro-F1: 0.8039 | Val MCC: 0.7854 | LR: 1.00e-03
✓ Best model saved.


...

Epoch 05 | Loss: 0.4586 | Val Acc: 0.8804 | Val Balanced Acc: 0.8302 | Val Macro-F1: 0.7899 | Val MCC: 0.7727 | LR: 1.00e-03
No improvement: 1/3


Epoch 6/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 06 | Loss: 0.4541 | Val Acc: 0.8804 | Val Balanced Acc: 0.8310 | Val Macro-F1: 0.7894 | Val MCC: 0.7731 | LR: 5.00e-04
No improvement: 2/3


Epoch 7/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 07 | Loss: 0.4541 | Val Acc: 0.8801 | Val Balanced Acc: 0.8309 | Val Macro-F1: 0.7888 | Val MCC: 0.7726 | LR: 5.00e-04
No improvement: 3/3

Early stopping triggered.


## Restoring best model

In [ ]:
tab_checkpoint = torch.load(

    best_tab_path,

    map_location=device
)


biomarker_model.load_state_dict(

    tab_checkpoint[
        "model_state_dict"
    ]
)


print(
    "Best epoch:",
    tab_checkpoint[
        "epoch"
    ]
)

print(
    "Best validation Macro-F1:",
    tab_checkpoint[
        "val_macro_f1"
    ]
)

Best epoch: 4
Best validation Macro-F1: 0.809383707137841


## Train / validation / test comparison

In [ ]:
train_tab_results = (
    evaluate_tabular_model(

        biomarker_model,

        train_tab_loader
    )
)


val_tab_results = (
    evaluate_tabular_model(

        biomarker_model,

        val_tab_loader
    )
)


test_tab_results = (
    evaluate_tabular_model(

        biomarker_model,

        test_tab_loader
    )
)


print(
    "=========== BIOMARKER MLP ONLY ==========="
)


print(
    f"Train Macro-F1: "
    f"{train_tab_results['macro_f1']:.4f}"
)


print(
    f"Validation Macro-F1: "
    f"{val_tab_results['macro_f1']:.4f}"
)


print(
    f"Test Macro-F1: "
    f"{test_tab_results['macro_f1']:.4f}"
)


print(
    f"Test Accuracy: "
    f"{test_tab_results['accuracy'] * 100:.2f}%"
)


print(
    f"Test Balanced Accuracy: "
    f"{test_tab_results['balanced_accuracy']:.4f}"
)


print(
    f"Test MCC: "
    f"{test_tab_results['mcc']:.4f}"
)

=========== BIOMARKER MLP ONLY ===========
Train Macro-F1: 0.8040
Validation Macro-F1: 0.8094
Test Macro-F1: 0.8135
Test Accuracy: 89.43%
Test Balanced Accuracy: 0.8345
Test MCC: 0.7969


## Classification Report

In [ ]:
print(

    classification_report(

        test_tab_results[
            "y_true"
        ],

        test_tab_results[
            "y_pred"
        ],

        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],

        digits=4,

        zero_division=0

    )

)

              precision    recall  f1-score   support

      Benign     0.9344    0.8588    0.8950      3435
  Pathogenic     0.5583    0.7143    0.6267       476
         VUS     0.9075    0.9305    0.9189      5726

    accuracy                         0.8943      9637
   macro avg     0.8001    0.8345    0.8135      9637
weighted avg     0.8999    0.8943    0.8959      9637



# Arm 1 + Arm 2

##  Creating Dataset

In [ ]:
class FusionDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tabular_matrix
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.tabular = torch.tensor(
            tabular_matrix,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            self.df["Label"].values,
            dtype=torch.long
        )


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        seq = encode_variant_sequence(
            row["seq201"],
            row["ReferenceAlleleVCF"],
            row["AlternateAlleleVCF"],
            window_size=101
        )

        seq = torch.tensor(
            seq,
            dtype=torch.float32
        )

        tab = self.tabular[idx]

        label = self.labels[idx]

        return (
            seq,
            tab,
            label
        )

In [ ]:
train_fusion_dataset = FusionDataset(
    train_df,
    X_train_tab
)

val_fusion_dataset = FusionDataset(
    val_df,
    X_val_tab
)

test_fusion_dataset = FusionDataset(
    test_df,
    X_test_tab
)


print(
    "Train:",
    len(train_fusion_dataset)
)

print(
    "Validation:",
    len(val_fusion_dataset)
)

print(
    "Test:",
    len(test_fusion_dataset)
)

Train: 80065
Validation: 10298
Test: 9637


## Dataloaders

In [ ]:
FUSION_BATCH_SIZE = (
    512
    if torch.cuda.is_available()
    else 256
)


train_fusion_loader = DataLoader(
    train_fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_fusion_loader = DataLoader(
    val_fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_fusion_loader = DataLoader(
    test_fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print(
    "Batch size:",
    FUSION_BATCH_SIZE
)

print(
    "Train batches:",
    len(train_fusion_loader)
)

Batch size: 256
Train batches: 313


## Fusion Model

In [ ]:
class GatedTwoArmFusionModel(nn.Module):

    def __init__(
        self,
        tabular_input_dim,
        num_classes=3
    ):

        super().__init__()

        # ====================================================
        # ARM 1 — SEQUENCE
        # ====================================================

        self.sequence_encoder = SequenceCNN(
            embedding_dim=64
        )


        # Reduce sequence representation:
        # 64 -> 32
        self.sequence_projection = nn.Sequential(

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.LayerNorm(
                32
            )
        )


        # ====================================================
        # ARM 2 — BIOMARKERS
        # ====================================================

        self.biomarker_encoder = BiomarkerMLP(

            input_dim=tabular_input_dim,

            embedding_dim=32
        )


        self.biomarker_norm = nn.LayerNorm(
            32
        )


        # ====================================================
        # LEARNED ALPHA GATE
        #
        # Input:
        # 32 sequence
        # +
        # 32 biomarker
        # =
        # 64
        #
        # Output:
        # one scalar alpha per variant
        # ====================================================

        self.gate = nn.Linear(
            64,
            1
        )


        # Start by trusting biomarkers more.
        #
        # sigmoid(-1.5) ≈ 0.18
        nn.init.zeros_(
            self.gate.weight
        )

        nn.init.constant_(
            self.gate.bias,
            -1.5
        )


        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Linear(
                32,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        seq,
        tab,
        return_alpha=False
    ):

        # ----------------------------------------------------
        # Sequence representation
        # ----------------------------------------------------

        seq_features = self.sequence_encoder(
            seq
        )

        seq_features = self.sequence_projection(
            seq_features
        )


        # ----------------------------------------------------
        # Biomarker representation
        # ----------------------------------------------------

        tab_features = self.biomarker_encoder(
            tab
        )

        tab_features = self.biomarker_norm(
            tab_features
        )


        # ----------------------------------------------------
        # Compute alpha
        # ----------------------------------------------------

        gate_input = torch.cat(
            [
                seq_features,
                tab_features
            ],
            dim=1
        )


        alpha = torch.sigmoid(
            self.gate(
                gate_input
            )
        )


        # ----------------------------------------------------
        # BIOMARKER-ANCHORED FUSION
        # ----------------------------------------------------

        fused = (
            tab_features
            +
            alpha
            *
            seq_features
        )


        logits = self.classifier(
            fused
        )


        if return_alpha:

            return (
                logits,
                alpha
            )


        return logits

In [ ]:
gated_model = GatedTwoArmFusionModel(

    tabular_input_dim=
        X_train_tab.shape[1],

    num_classes=3

).to(device)

## Loading pretrained sequence encoder

In [ ]:
sequence_checkpoint = torch.load(
    best_model_path,
    map_location=device
)


sequence_state = {

    key.replace(
        "encoder.",
        "",
        1
    ): value

    for key, value in
    sequence_checkpoint[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}


gated_model.sequence_encoder.load_state_dict(
    sequence_state
)


print(
    "✅ Sequence encoder restored."
)

✅ Sequence encoder restored.


## Loading biomarker encoder

In [ ]:
biomarker_checkpoint = torch.load(
    best_tab_path,
    map_location=device
)


biomarker_state = {

    key.replace(
        "encoder.",
        "",
        1
    ): value

    for key, value in
    biomarker_checkpoint[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}


gated_model.biomarker_encoder.load_state_dict(
    biomarker_state
)


print(
    "✅ Biomarker encoder restored."
)

✅ Biomarker encoder restored.


## Checking Starting point of Alpha gate

In [ ]:
seq_batch, tab_batch, labels = next(
    iter(train_fusion_loader)
)


seq_batch = seq_batch.to(
    device
)

tab_batch = tab_batch.to(
    device
)


with torch.no_grad():

    logits, alpha = gated_model(

        seq_batch,

        tab_batch,

        return_alpha=True
    )


print(
    "Output shape:",
    logits.shape
)

print(
    "Alpha shape:",
    alpha.shape
)

print(
    "Mean alpha:",
    alpha.mean().item()
)

print(
    "Minimum alpha:",
    alpha.min().item()
)

print(
    "Maximum alpha:",
    alpha.max().item()
)

Output shape: torch.Size([256, 3])
Alpha shape: torch.Size([256, 1])
Mean alpha: 0.18242551386356354
Minimum alpha: 0.18242552876472473
Maximum alpha: 0.18242552876472473


## Stage 1: train only gate + projection + classifier

In [ ]:
for param in gated_model.sequence_encoder.parameters():
    param.requires_grad = False

for param in gated_model.biomarker_encoder.parameters():
    param.requires_grad = False

### Optimizer

In [ ]:
gate_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        gated_model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

gate_loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)

### Training

In [ ]:
def train_gated_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    loss_fn,
    epochs,
    stage_name,
    checkpoint_path
):

    best_val_f1 = -1.0
    no_improvement = 0
    patience = 3

    for epoch in range(epochs):

        # ============================
        # TRAIN
        # ============================

        model.train()

        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"{stage_name} {epoch+1}/{epochs}"
        )

        for seq, tab, targets in progress:

            seq = seq.to(device)
            tab = tab.to(device)
            targets = targets.to(device)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                seq,
                tab
            )

            loss = loss_fn(
                logits,
                targets
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            optimizer.step()

            running_loss += (
                loss.item()
                * seq.size(0)
            )

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )


        train_loss = (
            running_loss
            /
            len(train_fusion_dataset)
        )


        # ============================
        # VALIDATION
        # ============================

        val_results = evaluate_fusion_model(
            model,
            val_loader
        )

        val_f1 = val_results[
            "macro_f1"
        ]


        # ============================
        # ALPHA STATISTICS
        # ============================

        model.eval()

        alpha_values = []

        with torch.no_grad():

            for seq, tab, _ in val_loader:

                seq = seq.to(device)
                tab = tab.to(device)

                _, alpha = model(
                    seq,
                    tab,
                    return_alpha=True
                )

                alpha_values.append(
                    alpha.cpu()
                )

        alpha_values = torch.cat(
            alpha_values,
            dim=0
        )


        mean_alpha = (
            alpha_values.mean().item()
        )

        std_alpha = (
            alpha_values.std().item()
        )


        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {train_loss:.4f} | "
            f"Val Acc: {val_results['accuracy']:.4f} | "
            f"Val Balanced Acc: {val_results['balanced_accuracy']:.4f} | "
            f"Val Macro-F1: {val_f1:.4f} | "
            f"Val MCC: {val_results['mcc']:.4f} | "
            f"Alpha Mean: {mean_alpha:.4f} | "
            f"Alpha Std: {std_alpha:.4f}"
        )


        # ============================
        # CHECKPOINT
        # ============================

        if val_f1 > best_val_f1 + 1e-4:

            best_val_f1 = val_f1
            no_improvement = 0

            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "epoch":
                        epoch + 1,

                    "val_macro_f1":
                        best_val_f1,

                    "mean_alpha":
                        mean_alpha,

                    "std_alpha":
                        std_alpha
                },
                checkpoint_path
            )

            print("✓ Best model saved.")

        else:

            no_improvement += 1

            print(
                f"No improvement: "
                f"{no_improvement}/{patience}"
            )


        if no_improvement >= patience:

            print(
                "\nEarly stopping triggered."
            )

            break

In [ ]:
for param in gated_model.sequence_encoder.parameters():
    param.requires_grad = False

for param in gated_model.biomarker_encoder.parameters():
    param.requires_grad = False


stage1_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        gated_model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)


train_gated_model(
    model=gated_model,
    train_loader=train_fusion_loader,
    val_loader=val_fusion_loader,
    optimizer=stage1_optimizer,
    loss_fn=gate_loss_fn,
    epochs=3,
    stage_name="Stage 1 Gate Warmup",
    checkpoint_path=
        CHECKPOINT_DIR /
        "gated_stage1_best.pt"
)

Stage 1 Gate Warmup 1/3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.4551 | Val Acc: 0.8713 | Val Balanced Acc: 0.8319 | Val Macro-F1: 0.7767 | Val MCC: 0.7597 | Alpha Mean: 0.5828 | Alpha Std: 0.1020
✓ Best model saved.


Stage 1 Gate Warmup 2/3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 02 | Loss: 0.4495 | Val Acc: 0.8845 | Val Balanced Acc: 0.8344 | Val Macro-F1: 0.7982 | Val MCC: 0.7797 | Alpha Mean: 0.5500 | Alpha Std: 0.1173
✓ Best model saved.


Stage 1 Gate Warmup 3/3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 03 | Loss: 0.4511 | Val Acc: 0.8617 | Val Balanced Acc: 0.8336 | Val Macro-F1: 0.7658 | Val MCC: 0.7462 | Alpha Mean: 0.6240 | Alpha Std: 0.1111
No improvement: 1/3


### Check Alpha

In [ ]:
seq_batch, tab_batch, labels = next(
    iter(val_fusion_loader)
)

seq_batch = seq_batch.to(device)
tab_batch = tab_batch.to(device)

gated_model.eval()

with torch.no_grad():

    logits, alpha = gated_model(
        seq_batch,
        tab_batch,
        return_alpha=True
    )

print("Mean alpha:", alpha.mean().item())
print("Min alpha:", alpha.min().item())
print("Max alpha:", alpha.max().item())
print("Std alpha:", alpha.std().item())

Mean alpha: 0.6808328628540039
Min alpha: 0.41472360491752625
Max alpha: 0.8438806533813477
Std alpha: 0.09222152084112167


In [ ]:
torch.save(
    checkpoint,
    CHECKPOINT_DIR / "gated_stage1_best.pt"
)

## Stage 2: Unfreeze Everything

In [ ]:
for param in gated_model.parameters():
    param.requires_grad = True

### Optimizer

In [ ]:
gate_optimizer = torch.optim.AdamW(
    [
        {
            "params":
                gated_model
                .sequence_encoder
                .parameters(),
            "lr": 5e-5
        },

        {
            "params":
                gated_model
                .biomarker_encoder
                .parameters(),
            "lr": 1e-4
        },

        {
            "params":
                gated_model
                .sequence_projection
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .gate
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .classifier
                .parameters(),
            "lr": 3e-4
        }
    ],

    weight_decay=1e-4
)

### Training

In [ ]:
stage1_checkpoint = torch.load(
    CHECKPOINT_DIR /
    "gated_stage1_best.pt",
    map_location=device
)

gated_model.load_state_dict(
    stage1_checkpoint[
        "model_state_dict"
    ]
)

<All keys matched successfully>

In [ ]:
for param in gated_model.parameters():
    param.requires_grad = True


stage2_optimizer = torch.optim.AdamW(
    [
        {
            "params":
                gated_model
                .sequence_encoder
                .parameters(),
            "lr": 5e-5
        },

        {
            "params":
                gated_model
                .biomarker_encoder
                .parameters(),
            "lr": 1e-4
        },

        {
            "params":
                gated_model
                .sequence_projection
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .gate
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .classifier
                .parameters(),
            "lr": 3e-4
        }
    ],
    weight_decay=1e-4
)


train_gated_model(
    model=gated_model,
    train_loader=train_fusion_loader,
    val_loader=val_fusion_loader,
    optimizer=stage2_optimizer,
    loss_fn=gate_loss_fn,
    epochs=10,
    stage_name="Stage 2 Fine-Tuning",
    checkpoint_path=
        CHECKPOINT_DIR /
        "gated_stage2_best.pt"
)

Stage 2 Fine-Tuning 1/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.4451 | Val Acc: 0.8669 | Val Balanced Acc: 0.8362 | Val Macro-F1: 0.7736 | Val MCC: 0.7537 | Alpha Mean: 0.5730 | Alpha Std: 0.1215
✓ Best model saved.


Stage 2 Fine-Tuning 2/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 02 | Loss: 0.4417 | Val Acc: 0.8549 | Val Balanced Acc: 0.8384 | Val Macro-F1: 0.7599 | Val MCC: 0.7377 | Alpha Mean: 0.5514 | Alpha Std: 0.1230
No improvement: 1/3


Stage 2 Fine-Tuning 3/10:   0%|          | 0/313 [00:00<?, ?it/s]

...

Epoch 05 | Loss: 0.4299 | Val Acc: 0.8609 | Val Balanced Acc: 0.8347 | Val Macro-F1: 0.7657 | Val MCC: 0.7452 | Alpha Mean: 0.6337 | Alpha Std: 0.1060
No improvement: 1/3


Stage 2 Fine-Tuning 6/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 06 | Loss: 0.4235 | Val Acc: 0.8618 | Val Balanced Acc: 0.8368 | Val Macro-F1: 0.7671 | Val MCC: 0.7469 | Alpha Mean: 0.6515 | Alpha Std: 0.1017
No improvement: 2/3


Stage 2 Fine-Tuning 7/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 07 | Loss: 0.4260 | Val Acc: 0.8589 | Val Balanced Acc: 0.8388 | Val Macro-F1: 0.7649 | Val MCC: 0.7430 | Alpha Mean: 0.6725 | Alpha Std: 0.0941
No improvement: 3/3

Early stopping triggered.


In [ ]:
stage2_checkpoint = torch.load(
    CHECKPOINT_DIR / "gated_stage2_best.pt",
    map_location=device
)

gated_model.load_state_dict(
    stage2_checkpoint["model_state_dict"]
)

print("Best epoch:",
      stage2_checkpoint["epoch"])

print("Best Val Macro-F1:",
      stage2_checkpoint["val_macro_f1"])

print("Mean alpha:",
      stage2_checkpoint["mean_alpha"])

print("Alpha std:",
      stage2_checkpoint["std_alpha"])

Best epoch: 4
Best Val Macro-F1: 0.7915132989945858
Mean alpha: 0.6232147812843323
Alpha std: 0.10979373008012772


## Testing

In [ ]:
test_gated_results = evaluate_fusion_model(
    gated_model,
    test_fusion_loader
)

print(
    "Test Accuracy:",
    test_gated_results["accuracy"]
)

print(
    "Test Balanced Accuracy:",
    test_gated_results["balanced_accuracy"]
)

print(
    "Test Macro-F1:",
    test_gated_results["macro_f1"]
)

print(
    "Test MCC:",
    test_gated_results["mcc"]
)

Test Accuracy: 0.8827435924042751
Test Balanced Accuracy: 0.834854527967645
Test Macro-F1: 0.794609912539816
Test MCC: 0.7780371971248816


## Classification Report

In [ ]:
print(
    classification_report(
        test_gated_results["y_true"],
        test_gated_results["y_pred"],
        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.9294    0.8664    0.8968      3435
  Pathogenic     0.4807    0.7332    0.5807       476
         VUS     0.9077    0.9050    0.9063      5726

    accuracy                         0.8827      9637
   macro avg     0.7726    0.8349    0.7946      9637
weighted avg     0.8943    0.8827    0.8869      9637



# Arm 3: GCN

## Installs

In [43]:
import sys
import subprocess
import torch

try:
    import torch_geometric
    print("PyG already installed:", torch_geometric.__version__)

except ImportError:

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch_geometric"
    ])

    import torch_geometric

    print(
        "Installed PyG:",
        torch_geometric.__version__
    )

Installed PyG: 2.8.0.post1


In [44]:
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GCNConv

print("✅ PyTorch Geometric ready")

✅ PyTorch Geometric ready


## Clean Graph features

In [45]:
GRAPH_NUMERIC_FEATURES = [
    "phylop",
    "phastcons",
    "Tab_Origin_Numeric",
    "Tab_Somatic_Impact"
]

GRAPH_CATEGORICAL_FEATURE = (
    "consequence_category"
)

print(
    "Graph numeric features:",
    GRAPH_NUMERIC_FEATURES
)

print(
    "Categorical:",
    GRAPH_CATEGORICAL_FEATURE
)

Graph numeric features: ['phylop', 'phastcons', 'Tab_Origin_Numeric', 'Tab_Somatic_Impact']
Categorical: consequence_category


## Building graph-node feature matrices

In [46]:
graph_train_medians = {}

for feature in GRAPH_NUMERIC_FEATURES:

    median = train_df[
        feature
    ].median()

    if pd.isna(median):
        median = 0.0

    graph_train_medians[
        feature
    ] = float(median)


print(
    graph_train_medians
)

{'phylop': 1.4950000047683716, 'phastcons': 0.9909999966621399, 'Tab_Origin_Numeric': 1.0, 'Tab_Somatic_Impact': 1.0}


In [47]:
graph_consequence_categories = sorted(

    train_df[
        GRAPH_CATEGORICAL_FEATURE
    ]
    .fillna("unknown")
    .astype(str)
    .unique()
)


print(
    graph_consequence_categories
)

['intron', 'missense', 'nonsense_or_stop', 'other', 'splice', 'synonymous', 'unknown', 'utr']


## Graph feature transformation

In [48]:
from sklearn.preprocessing import StandardScaler


def build_graph_feature_matrix(
    dataframe,
    fit_scaler=False,
    scaler=None
):

    temp = (
        dataframe
        .reset_index(drop=True)
        .copy()
    )


    # ==============================================
    # NUMERIC
    # ==============================================

    numeric = temp[
        GRAPH_NUMERIC_FEATURES
    ].copy()


    # Conservation missingness
    for feature in [
        "phylop",
        "phastcons"
    ]:

        numeric[
            f"{feature}_missing"
        ] = (
            temp[feature]
            .isna()
            .astype(np.float32)
        )


    # Training-derived imputation
    for feature in GRAPH_NUMERIC_FEATURES:

        numeric[feature] = (
            numeric[feature]
            .fillna(
                graph_train_medians[
                    feature
                ]
            )
        )


    numeric = numeric.astype(
        np.float32
    )


    # ==============================================
    # SCALE
    # ==============================================

    if fit_scaler:

        scaler = StandardScaler()

        numeric_scaled = (
            scaler.fit_transform(
                numeric
            )
        )

    else:

        numeric_scaled = (
            scaler.transform(
                numeric
            )
        )


    # ==============================================
    # CONSEQUENCE ONE-HOT
    # ==============================================

    values = (
        temp[
            GRAPH_CATEGORICAL_FEATURE
        ]
        .fillna("unknown")
        .astype(str)
    )


    cat_to_idx = {

        category: i

        for i, category
        in enumerate(
            graph_consequence_categories
        )
    }


    categorical = np.zeros(
        (
            len(temp),
            len(
                graph_consequence_categories
            )
        ),
        dtype=np.float32
    )


    for row_idx, value in enumerate(values):

        if value in cat_to_idx:

            categorical[
                row_idx,
                cat_to_idx[value]
            ] = 1.0


    # ==============================================
    # FINAL
    # ==============================================

    result = np.concatenate(
        [
            numeric_scaled.astype(
                np.float32
            ),
            categorical
        ],
        axis=1
    )


    return result, scaler

## Building graph

In [49]:
K_SAME_CONSEQUENCE = 4

K_CONSERVATION = 3

K_PROX = 2

MAX_BP = 5000

### Feature matrices

In [50]:
X_train_graph, graph_scaler = (
    build_graph_feature_matrix(
        train_df,
        fit_scaler=True
    )
)


X_val_graph, _ = (
    build_graph_feature_matrix(
        val_df,
        fit_scaler=False,
        scaler=graph_scaler
    )
)


X_test_graph, _ = (
    build_graph_feature_matrix(
        test_df,
        fit_scaler=False,
        scaler=graph_scaler
    )
)


print(
    "Train:",
    X_train_graph.shape
)

print(
    "Validation:",
    X_val_graph.shape
)

print(
    "Test:",
    X_test_graph.shape
)


assert np.isfinite(
    X_train_graph
).all()

assert np.isfinite(
    X_val_graph
).all()

assert np.isfinite(
    X_test_graph
).all()


print(
    "✅ Graph node features ready"
)

Train: (80065, 14)
Validation: (10298, 14)
Test: (9637, 14)
✅ Graph node features ready


### Graph Edges

In [51]:
def build_variant_graph_edges_v2(
    dataframe,
    k_same_consequence=4,
    k_conservation=3,
    k_prox=2,
    max_bp=5000
):

    temp = (
        dataframe
        .reset_index(drop=True)
        .copy()
    )

    edges = set()


    # ========================================================
    # BASIC CLEANUP
    # ========================================================

    temp["_gene"] = (
        temp["GeneSymbol"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    temp["_consequence"] = (
        temp["consequence_category"]
        .fillna("unknown")
        .astype(str)
    )


    # Use TRAIN-derived medians already calculated earlier
    temp["_phylop"] = (
        temp["phylop"]
        .fillna(
            graph_train_medians["phylop"]
        )
        .astype(float)
    )

    temp["_phastcons"] = (
        temp["phastcons"]
        .fillna(
            graph_train_medians["phastcons"]
        )
        .astype(float)
    )


    valid_gene = (
        (temp["_gene"] != "")
        &
        (temp["_gene"] != "-")
        &
        (temp["_gene"].str.lower() != "nan")
    )


    # ========================================================
    # HELPER
    # ========================================================

    def add_edge(i, j):

        i = int(i)
        j = int(j)

        if i == j:
            return

        edges.add((i, j))
        edges.add((j, i))


    # ========================================================
    # PROCESS EACH GENE SEPARATELY
    # ========================================================

    for gene, gene_group in (
        temp.loc[valid_gene]
        .groupby(
            "_gene",
            sort=False
        )
    ):

        if len(gene_group) < 2:
            continue


        # ----------------------------------------------------
        # A. SAME GENE + SAME CONSEQUENCE
        # ----------------------------------------------------

        for consequence, subgroup in (
            gene_group.groupby(
                "_consequence",
                sort=False
            )
        ):

            if len(subgroup) < 2:
                continue


            subgroup = subgroup.sort_values(
                "Start"
            )


            indices = (
                subgroup.index.to_numpy()
            )

            positions = (
                subgroup["Start"]
                .to_numpy()
            )


            for local_i in range(
                len(subgroup)
            ):

                distances = []


                for local_j in range(
                    len(subgroup)
                ):

                    if local_i == local_j:
                        continue


                    distance = abs(
                        positions[local_i]
                        -
                        positions[local_j]
                    )


                    distances.append(
                        (
                            distance,
                            indices[local_j]
                        )
                    )


                distances.sort(
                    key=lambda x: x[0]
                )


                for _, neighbor in (
                    distances[
                        :k_same_consequence
                    ]
                ):

                    add_edge(
                        indices[local_i],
                        neighbor
                    )


        # ----------------------------------------------------
        # B. SAME GENE + SIMILAR CONSERVATION
        # ----------------------------------------------------

        gene_indices = (
            gene_group.index.to_numpy()
        )


        conservation = (
            gene_group[
                [
                    "_phylop",
                    "_phastcons"
                ]
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        # Normalize locally so PhyloP does not dominate
        means = conservation.mean(
            axis=0,
            keepdims=True
        )

        stds = conservation.std(
            axis=0,
            keepdims=True
        )

        stds[
            stds < 1e-6
        ] = 1.0


        conservation_norm = (
            conservation - means
        ) / stds


        for i in range(
            len(gene_group)
        ):

            distances = []


            for j in range(
                len(gene_group)
            ):

                if i == j:
                    continue


                distance = np.linalg.norm(
                    conservation_norm[i]
                    -
                    conservation_norm[j]
                )


                distances.append(
                    (
                        distance,
                        gene_indices[j]
                    )
                )


            distances.sort(
                key=lambda x: x[0]
            )


            for _, neighbor in (
                distances[
                    :k_conservation
                ]
            ):

                add_edge(
                    gene_indices[i],
                    neighbor
                )


        # ----------------------------------------------------
        # C. GENOMIC PROXIMITY — SAME GENE ONLY
        # ----------------------------------------------------

        positional = gene_group.sort_values(
            "Start"
        )


        indices = positional.index.to_numpy()

        positions = (
            positional["Start"]
            .to_numpy()
        )


        m = len(positional)


        for local_i in range(m):

            candidates = []


            # Look around locally rather than comparing
            # every pair
            for offset in range(
                1,
                k_prox + 3
            ):

                left = (
                    local_i - offset
                )

                right = (
                    local_i + offset
                )


                if left >= 0:

                    distance = (
                        positions[local_i]
                        -
                        positions[left]
                    )

                    if distance <= max_bp:

                        candidates.append(
                            (
                                distance,
                                indices[left]
                            )
                        )


                if right < m:

                    distance = (
                        positions[right]
                        -
                        positions[local_i]
                    )

                    if distance <= max_bp:

                        candidates.append(
                            (
                                distance,
                                indices[right]
                            )
                        )


            candidates.sort(
                key=lambda x: x[0]
            )


            for _, neighbor in (
                candidates[:k_prox]
            ):

                add_edge(
                    indices[local_i],
                    neighbor
                )


    # ========================================================
    # CONVERT TO edge_index
    # ========================================================

    if len(edges) == 0:

        return torch.empty(
            (2, 0),
            dtype=torch.long
        )


    edge_array = np.asarray(
        list(edges),
        dtype=np.int64
    )


    edge_index = torch.tensor(
        edge_array.T,
        dtype=torch.long
    )


    return edge_index

### Builidng train/test/val graphs seperately

In [52]:
print(
    "Building Graph v2..."
)


train_edge_index_v2 = (
    build_variant_graph_edges_v2(
        train_df,
        k_same_consequence=
            K_SAME_CONSEQUENCE,
        k_conservation=
            K_CONSERVATION,
        k_prox=
            K_PROX,
        max_bp=
            MAX_BP
    )
)


val_edge_index_v2 = (
    build_variant_graph_edges_v2(
        val_df,
        k_same_consequence=
            K_SAME_CONSEQUENCE,
        k_conservation=
            K_CONSERVATION,
        k_prox=
            K_PROX,
        max_bp=
            MAX_BP
    )
)


test_edge_index_v2 = (
    build_variant_graph_edges_v2(
        test_df,
        k_same_consequence=
            K_SAME_CONSEQUENCE,
        k_conservation=
            K_CONSERVATION,
        k_prox=
            K_PROX,
        max_bp=
            MAX_BP
    )
)


print(
    "Train directed edges:",
    train_edge_index_v2.shape[1]
)

print(
    "Val directed edges:",
    val_edge_index_v2.shape[1]
)

print(
    "Test directed edges:",
    test_edge_index_v2.shape[1]
)

Building Graph v2...
Train directed edges: 532370
Val directed edges: 69356
Test directed edges: 63530


### Sanity check

In [53]:
def graph_stats(
    name,
    dataframe,
    edge_index
):

    n_nodes = len(
        dataframe
    )

    n_edges = (
        edge_index.shape[1]
    )

    avg_degree = (
        n_edges
        /
        max(
            n_nodes,
            1
        )
    )


    isolated_estimate = None


    if n_edges > 0:

        connected_nodes = torch.unique(
            edge_index
        ).numel()

        isolated_estimate = (
            n_nodes
            -
            connected_nodes
        )


    print(
        f"{name}"
    )

    print(
        "Nodes:",
        n_nodes
    )

    print(
        "Directed edges:",
        n_edges
    )

    print(
        "Average degree:",
        round(
            avg_degree,
            2
        )
    )

    print(
        "Estimated isolated nodes:",
        isolated_estimate
    )

    print()


graph_stats(
    "TRAIN V2",
    train_df,
    train_edge_index_v2
)

graph_stats(
    "VALIDATION V2",
    val_df,
    val_edge_index_v2
)

graph_stats(
    "TEST V2",
    test_df,
    test_edge_index_v2
)

TRAIN V2
Nodes: 80065
Directed edges: 532370
Average degree: 6.65
Estimated isolated nodes: 3087

VALIDATION V2
Nodes: 10298
Directed edges: 69356
Average degree: 6.73
Estimated isolated nodes: 386

TEST V2
Nodes: 9637
Directed edges: 63530
Average degree: 6.59
Estimated isolated nodes: 403



## PyG Data Objects

In [59]:
train_graph_v2 = Data(

    x=torch.tensor(
        X_train_graph,
        dtype=torch.float32
    ),

    edge_index=
        train_edge_index_v2,

    y=torch.tensor(
        train_df["Label"].values,
        dtype=torch.long
    )
)


val_graph_v2 = Data(

    x=torch.tensor(
        X_val_graph,
        dtype=torch.float32
    ),

    edge_index=
        val_edge_index_v2,

    y=torch.tensor(
        val_df["Label"].values,
        dtype=torch.long
    )
)


test_graph_v2 = Data(

    x=torch.tensor(
        X_test_graph,
        dtype=torch.float32
    ),

    edge_index=
        test_edge_index_v2,

    y=torch.tensor(
        test_df["Label"].values,
        dtype=torch.long
    )
)

In [60]:
train_graph_v2 = (
    train_graph_v2.to(device)
)

val_graph_v2 = (
    val_graph_v2.to(device)
)

test_graph_v2 = (
    test_graph_v2.to(device)
)

print(
    "✅ Graph v2 ready"
)

✅ Graph v2 ready


## GCN Model

In [61]:
class VariantGCN(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        embedding_dim=32
    ):

        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim
        )

        self.norm1 = nn.BatchNorm1d(
            hidden_dim
        )

        self.conv2 = GCNConv(
            hidden_dim,
            embedding_dim
        )

        self.norm2 = nn.BatchNorm1d(
            embedding_dim
        )

        self.dropout = nn.Dropout(
            0.30
        )


    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = self.norm1(x)

        x = F.relu(x)

        x = self.dropout(x)


        x = self.conv2(
            x,
            edge_index
        )

        x = self.norm2(x)

        x = F.relu(x)

        return x

In [62]:
class GCNOnlyModel(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes=3
    ):

        super().__init__()

        self.encoder = VariantGCN(
            input_dim=input_dim,
            hidden_dim=32,
            embedding_dim=32
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                32,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        x,
        edge_index
    ):

        embeddings = self.encoder(
            x,
            edge_index
        )

        logits = self.classifier(
            embeddings
        )

        return logits

In [63]:
gcn_model_v2 = GCNOnlyModel(

    input_dim=
        X_train_graph.shape[1],

    num_classes=3

).to(device)


gcn_loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)


gcn_optimizer_v2 = (
    torch.optim.AdamW(

        gcn_model_v2.parameters(),

        lr=1e-3,

        weight_decay=1e-4
    )
)


gcn_scheduler_v2 = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(

        gcn_optimizer_v2,

        mode="max",

        factor=0.5,

        patience=2
    )
)


print(
    gcn_model_v2
)


print(
    "\nParameters:",
    sum(
        p.numel()
        for p
        in gcn_model_v2.parameters()
        if p.requires_grad
    )
)

GCNOnlyModel(
  (encoder): VariantGCN(
    (conv1): GCNConv(14, 32)
    (norm1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): GCNConv(32, 32)
    (norm2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)

Parameters: 2819


## Evaluation Function

In [ ]:
@torch.no_grad()
def evaluate_gcn_fullgraph(
    model,
    graph
):

    model.eval()

    logits = model(
        graph.x,
        graph.edge_index
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    predictions = torch.argmax(
        probabilities,
        dim=1
    )

    y_true = (
        graph.y
        .cpu()
        .numpy()
    )

    y_pred = (
        predictions
        .cpu()
        .numpy()
    )

    y_prob = (
        probabilities
        .cpu()
        .numpy()
    )

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }

## Training loop

In [ ]:
GCN_V2_EPOCHS = 40

GCN_V2_PATIENCE = 6


best_gcn_v2_f1 = -1.0

no_improvement = 0


best_gcn_v2_path = (
    CHECKPOINT_DIR /
    "gcn_v2_best.pt"
)


for epoch in range(
    GCN_V2_EPOCHS
):

    # ==========================================
    # TRAIN
    # ==========================================

    gcn_model_v2.train()

    gcn_optimizer_v2.zero_grad(
        set_to_none=True
    )


    logits = gcn_model_v2(

        train_graph_v2.x,

        train_graph_v2.edge_index
    )


    loss = gcn_loss_fn(

        logits,

        train_graph_v2.y
    )


    loss.backward()


    torch.nn.utils.clip_grad_norm_(

        gcn_model_v2.parameters(),

        5.0
    )


    gcn_optimizer_v2.step()


    # ==========================================
    # VALIDATION
    # ==========================================

    val_results = (
        evaluate_gcn_fullgraph(

            gcn_model_v2,

            val_graph_v2
        )
    )


    val_f1 = (
        val_results["macro_f1"]
    )


    gcn_scheduler_v2.step(
        val_f1
    )


    current_lr = (
        gcn_optimizer_v2
        .param_groups[0]["lr"]
    )


    print(

        f"Epoch {epoch+1:02d} | "

        f"Loss: "
        f"{loss.item():.4f} | "

        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "

        f"Val Bal Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "

        f"Val Macro-F1: "
        f"{val_f1:.4f} | "

        f"Val MCC: "
        f"{val_results['mcc']:.4f} | "

        f"LR: "
        f"{current_lr:.2e}"
    )


    if (
        val_f1
        >
        best_gcn_v2_f1
        + 1e-4
    ):

        best_gcn_v2_f1 = (
            val_f1
        )

        no_improvement = 0


        torch.save(
            {
                "model_state_dict":
                    gcn_model_v2.state_dict(),

                "epoch":
                    epoch + 1,

                "val_macro_f1":
                    best_gcn_v2_f1
            },

            best_gcn_v2_path
        )


        print(
            "✓ Best Graph v2 model saved."
        )


    else:

        no_improvement += 1

        print(
            f"No improvement: "
            f"{no_improvement}/"
            f"{GCN_V2_PATIENCE}"
        )


    if (
        no_improvement
        >=
        GCN_V2_PATIENCE
    ):

        print(
            "\nEarly stopping."
        )

        break

Epoch 01 | Loss: 0.8146 | Val Acc: 0.6808 | Val Bal Acc: 0.6722 | Val Macro-F1: 0.5824 | Val MCC: 0.4849 | LR: 1.00e-03
✓ Best Graph v2 model saved.
Epoch 02 | Loss: 0.8107 | Val Acc: 0.6868 | Val Bal Acc: 0.6756 | Val Macro-F1: 0.5872 | Val MCC: 0.4914 | LR: 1.00e-03
✓ Best Graph v2 model saved.
Epoch 03 | Loss: 0.8068 | Val Acc: 0.6922 | Val Bal Acc: 0.6774 | Val Macro-F1: 0.5912 | Val MCC: 0.4966 | LR: 1.00e-03
✓ Best Graph v2 model saved.
Epoch 04 | Loss: 0.8006 | Val Acc: 0.6999 | Val Bal Acc: 0.6804 | Val Macro-F1: 0.5970 | Val MCC: 0.5046 | LR: 1.00e-03
✓ Best Graph v2 model saved.
Epoch 05 | Loss: 0.7937 | Val Acc: 0.7065 | Val Bal Acc: 0.6836 | Val Macro-F1: 0.6026 | Val MCC: 0.5107 | LR: 1.00e-03
✓ Best Graph v2 model saved.
Epoch 06 | Loss: 0.7894 | Val Acc: 0.7116 | Val Bal Acc: 0.6863 | Val Macro-F1: 0.6070 | Val MCC: 0.5157 | LR: 1.00e-03
✓ Best Graph v2 model saved.
Epoch 07 | Loss: 0.7875 | Val Acc: 0.7166 | Val Bal Acc: 0.6845 | Val Macro-F1: 0.6097 | Val MCC: 0.5198 |

## Final evaluation

In [ ]:
test_gcn_v2 = (
    evaluate_gcn_fullgraph(
        gcn_model_v2,
        test_graph_v2
    )
)


# print(
#     "Graph v1 Macro-F1:",
#     test_gcn_results[
#         "macro_f1"
#     ]
# )

print(
    "Graph v2 Macro-F1:",
    test_gcn_v2[
        "macro_f1"
    ]
)

print(
    "Graph v2 Balanced Acc:",
    test_gcn_v2[
        "balanced_accuracy"
    ]
)

print(
    "Graph v2 Accuracy:",
    test_gcn_v2[
        "accuracy"
    ]
)

print(
    "Graph v2 MCC:",
    test_gcn_v2[
        "mcc"
    ]
)

Graph v1 Macro-F1: 0.47298042128098966
Graph v2 Macro-F1: 0.6811191827519218
Graph v2 Balanced Acc: 0.720568687075589
Graph v2 Accuracy: 0.7905987340458649
Graph v2 MCC: 0.6125117678618928


## Classification report

In [ ]:
print(
    classification_report(

        test_gcn_v2[
            "y_true"
        ],

        test_gcn_v2[
            "y_pred"
        ],

        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],

        digits=4,

        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.7777    0.8015    0.7894      3435
  Pathogenic     0.3376    0.5567    0.4203       476
         VUS     0.8662    0.8035    0.8337      5726

    accuracy                         0.7906      9637
   macro avg     0.6605    0.7206    0.6811      9637
weighted avg     0.8085    0.7906    0.7975      9637



# Gated MLP + GCN Model

In [ ]:
class GatedMLPGCNModel(nn.Module):

    def __init__(
        self,
        tabular_input_dim,
        graph_input_dim,
        num_classes=3
    ):

        super().__init__()

        # =========================================
        # BIOMARKER ARM
        # =========================================

        self.biomarker_encoder = BiomarkerMLP(
            input_dim=tabular_input_dim,
            embedding_dim=32
        )

        self.biomarker_norm = nn.LayerNorm(32)


        # =========================================
        # GRAPH ARM
        # =========================================

        self.graph_encoder = VariantGCN(
            input_dim=graph_input_dim,
            hidden_dim=32,
            embedding_dim=32
        )

        self.graph_projection = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.LayerNorm(32)
        )


        # =========================================
        # BETA GATE
        # =========================================

        self.gate = nn.Linear(
            64,
            1
        )

        nn.init.zeros_(
            self.gate.weight
        )

        # sigmoid(-1.5) ≈ 0.18
        nn.init.constant_(
            self.gate.bias,
            -1.5
        )


        # =========================================
        # CLASSIFIER
        # =========================================

        self.classifier = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(32, num_classes)
        )


    def forward(
        self,
        tab,
        graph_x,
        edge_index,
        return_beta=False
    ):

        # Biomarker embedding
        tab_features = self.biomarker_encoder(
            tab
        )

        tab_features = self.biomarker_norm(
            tab_features
        )


        # Graph embedding
        graph_features = self.graph_encoder(
            graph_x,
            edge_index
        )

        graph_features = self.graph_projection(
            graph_features
        )


        # Gate
        gate_input = torch.cat(
            [
                tab_features,
                graph_features
            ],
            dim=1
        )

        beta = torch.sigmoid(
            self.gate(
                gate_input
            )
        )


        # Biomarker-anchored fusion
        fused = (
            tab_features
            +
            beta * graph_features
        )


        logits = self.classifier(
            fused
        )


        if return_beta:
            return logits, beta

        return logits

In [ ]:
mlp_gcn_model = GatedMLPGCNModel(
    tabular_input_dim=X_train_tab.shape[1],
    graph_input_dim=X_train_graph.shape[1],
    num_classes=3
).to(device)

## Load presaved models

In [ ]:
biomarker_checkpoint = torch.load(
    best_tab_path,
    map_location=device
)

biomarker_state = {
    key.replace(
        "encoder.",
        "",
        1
    ): value

    for key, value in
    biomarker_checkpoint[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}

mlp_gcn_model.biomarker_encoder.load_state_dict(
    biomarker_state
)

print(
    "✅ Biomarker encoder loaded"
)

✅ Biomarker encoder loaded


In [ ]:
gcn_checkpoint = torch.load(
    best_gcn_v2_path,
    map_location=device
)

gcn_state = {
    key.replace(
        "encoder.",
        "",
        1
    ): value

    for key, value in
    gcn_checkpoint[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}

mlp_gcn_model.graph_encoder.load_state_dict(
    gcn_state
)

print(
    "✅ GCN encoder loaded"
)

✅ GCN encoder loaded


## Alignment check

In [ ]:
assert len(train_df) == train_graph_v2.num_nodes
assert len(val_df) == val_graph_v2.num_nodes
assert len(test_df) == test_graph_v2.num_nodes

assert X_train_tab.shape[0] == train_graph_v2.num_nodes
assert X_val_tab.shape[0] == val_graph_v2.num_nodes
assert X_test_tab.shape[0] == test_graph_v2.num_nodes

print(
    "✅ Biomarker rows and graph nodes aligned"
)

✅ Biomarker rows and graph nodes aligned


## Biomarkers to tensors

In [ ]:
train_tab_tensor = torch.tensor(
    X_train_tab,
    dtype=torch.float32,
    device=device
)

val_tab_tensor = torch.tensor(
    X_val_tab,
    dtype=torch.float32,
    device=device
)

test_tab_tensor = torch.tensor(
    X_test_tab,
    dtype=torch.float32,
    device=device
)

## Check initial β

In [ ]:
mlp_gcn_model.eval()

with torch.no_grad():

    logits, beta = mlp_gcn_model(
        train_tab_tensor,
        train_graph_v2.x,
        train_graph_v2.edge_index,
        return_beta=True
    )

print(
    "Output:",
    logits.shape
)

print(
    "Mean beta:",
    beta.mean().item()
)

print(
    "Min beta:",
    beta.min().item()
)

print(
    "Max beta:",
    beta.max().item()
)

Output: torch.Size([80065, 3])
Mean beta: 0.19920049607753754
Min beta: 0.18014387786388397
Max beta: 0.2120836228132248


## Stage 1: train gate only

In [ ]:
for param in (
    mlp_gcn_model
    .biomarker_encoder
    .parameters()
):
    param.requires_grad = False


for param in (
    mlp_gcn_model
    .graph_encoder
    .parameters()
):
    param.requires_grad = False

In [ ]:
stage1_mg_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        mlp_gcn_model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

mg_loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)

In [ ]:
@torch.no_grad()
def evaluate_mlp_gcn(
    model,
    tab_tensor,
    graph
):

    model.eval()

    logits, beta = model(
        tab_tensor,
        graph.x,
        graph.edge_index,
        return_beta=True
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    predictions = torch.argmax(
        probabilities,
        dim=1
    )

    y_true = graph.y.cpu().numpy()
    y_pred = predictions.cpu().numpy()

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "beta_mean":
            beta.mean().item(),

        "beta_std":
            beta.std().item(),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            probabilities
            .cpu()
            .numpy()
    }

In [ ]:
for epoch in range(3):

    mlp_gcn_model.train()

    stage1_mg_optimizer.zero_grad(
        set_to_none=True
    )

    logits = mlp_gcn_model(
        train_tab_tensor,
        train_graph_v2.x,
        train_graph_v2.edge_index
    )

    loss = mg_loss_fn(
        logits,
        train_graph_v2.y
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        mlp_gcn_model.parameters(),
        5.0
    )

    stage1_mg_optimizer.step()


    val_results = evaluate_mlp_gcn(
        mlp_gcn_model,
        val_tab_tensor,
        val_graph_v2
    )


    print(
        f"Stage 1 Epoch {epoch+1} | "
        f"Loss: {loss.item():.4f} | "
        f"Val Macro-F1: "
        f"{val_results['macro_f1']:.4f} | "
        f"Val Bal Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "
        f"Beta Mean: "
        f"{val_results['beta_mean']:.4f} | "
        f"Beta Std: "
        f"{val_results['beta_std']:.4f}"
    )

Stage 1 Epoch 1 | Loss: 0.9870 | Val Macro-F1: 0.3024 | Val Bal Acc: 0.6318 | Beta Mean: 0.2044 | Beta Std: 0.0081
Stage 1 Epoch 2 | Loss: 0.9645 | Val Macro-F1: 0.3027 | Val Bal Acc: 0.6328 | Beta Mean: 0.2096 | Beta Std: 0.0092
Stage 1 Epoch 3 | Loss: 0.9407 | Val Macro-F1: 0.3350 | Val Bal Acc: 0.6464 | Beta Mean: 0.2148 | Beta Std: 0.0103


In [ ]:
torch.save(
    {
        "model_state_dict":
            mlp_gcn_model.state_dict()
    },
    CHECKPOINT_DIR /
    "mlp_gcn_stage1.pt"
)

## Stage 2: fine tuning everything

In [ ]:
for param in mlp_gcn_model.parameters():
    param.requires_grad = True

In [ ]:
stage2_mg_optimizer = torch.optim.AdamW(
    [
        {
            "params":
                mlp_gcn_model
                .biomarker_encoder
                .parameters(),
            "lr": 1e-4
        },

        {
            "params":
                mlp_gcn_model
                .graph_encoder
                .parameters(),
            "lr": 5e-5
        },

        {
            "params":
                mlp_gcn_model
                .graph_projection
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                mlp_gcn_model
                .gate
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                mlp_gcn_model
                .classifier
                .parameters(),
            "lr": 3e-4
        }
    ],
    weight_decay=1e-4
)

In [ ]:
MG_EPOCHS = 20
MG_PATIENCE = 4

best_mg_f1 = -1.0
mg_no_improvement = 0

best_mg_path = (
    CHECKPOINT_DIR /
    "mlp_gcn_best.pt"
)


for epoch in range(MG_EPOCHS):

    mlp_gcn_model.train()

    stage2_mg_optimizer.zero_grad(
        set_to_none=True
    )


    logits = mlp_gcn_model(
        train_tab_tensor,
        train_graph_v2.x,
        train_graph_v2.edge_index
    )


    loss = mg_loss_fn(
        logits,
        train_graph_v2.y
    )


    loss.backward()


    torch.nn.utils.clip_grad_norm_(
        mlp_gcn_model.parameters(),
        5.0
    )


    stage2_mg_optimizer.step()


    val_results = evaluate_mlp_gcn(
        mlp_gcn_model,
        val_tab_tensor,
        val_graph_v2
    )


    val_f1 = val_results[
        "macro_f1"
    ]


    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {loss.item():.4f} | "
        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "
        f"Val Bal Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "
        f"Val Macro-F1: "
        f"{val_f1:.4f} | "
        f"Val MCC: "
        f"{val_results['mcc']:.4f} | "
        f"Beta Mean: "
        f"{val_results['beta_mean']:.4f} | "
        f"Beta Std: "
        f"{val_results['beta_std']:.4f}"
    )


    if val_f1 > best_mg_f1 + 1e-4:

        best_mg_f1 = val_f1
        mg_no_improvement = 0

        torch.save(
            {
                "model_state_dict":
                    mlp_gcn_model
                    .state_dict(),

                "epoch":
                    epoch + 1,

                "val_macro_f1":
                    best_mg_f1,

                "beta_mean":
                    val_results[
                        "beta_mean"
                    ],

                "beta_std":
                    val_results[
                        "beta_std"
                    ]
            },

            best_mg_path
        )

        print(
            "✓ Best MLP+GCN model saved."
        )

    else:

        mg_no_improvement += 1

        print(
            f"No improvement: "
            f"{mg_no_improvement}/"
            f"{MG_PATIENCE}"
        )


    if mg_no_improvement >= MG_PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break

Epoch 01 | Loss: 0.7842 | Val Acc: 0.6948 | Val Bal Acc: 0.7853 | Val Macro-F1: 0.6210 | Val MCC: 0.5568 | Beta Mean: 0.2489 | Beta Std: 0.0178
✓ Best MLP+GCN model saved.
Epoch 02 | Loss: 0.7771 | Val Acc: 0.7068 | Val Bal Acc: 0.7903 | Val Macro-F1: 0.6308 | Val MCC: 0.5688 | Beta Mean: 0.2507 | Beta Std: 0.0182
✓ Best MLP+GCN model saved.
Epoch 03 | Loss: 0.7734 | Val Acc: 0.7177 | Val Bal Acc: 0.7951 | Val Macro-F1: 0.6400 | Val MCC: 0.5792 | Beta Mean: 0.2524 | Beta Std: 0.0186
✓ Best MLP+GCN model saved.
Epoch 04 | Loss: 0.7670 | Val Acc: 0.7302 | Val Bal Acc: 0.8003 | Val Macro-F1: 0.6504 | Val MCC: 0.5915 | Beta Mean: 0.2541 | Beta Std: 0.0190
✓ Best MLP+GCN model saved.
Epoch 05 | Loss: 0.7596 | Val Acc: 0.7404 | Val Bal Acc: 0.8038 | Val Macro-F1: 0.6586 | Val MCC: 0.6021 | Beta Mean: 0.2559 | Beta Std: 0.0195
✓ Best MLP+GCN model saved.
Epoch 06 | Loss: 0.7547 | Val Acc: 0.7513 | Val Bal Acc: 0.8072 | Val Macro-F1: 0.6674 | Val MCC: 0.6134 | Beta Mean: 0.2576 | Beta Std: 0.0

In [ ]:
mg_checkpoint = torch.load(
    best_mg_path,
    map_location=device
)

mlp_gcn_model.load_state_dict(
    mg_checkpoint[
        "model_state_dict"
    ]
)

print(
    "Best epoch:",
    mg_checkpoint["epoch"]
)

print(
    "Best validation Macro-F1:",
    mg_checkpoint["val_macro_f1"]
)

print(
    "Beta mean:",
    mg_checkpoint["beta_mean"]
)

print(
    "Beta std:",
    mg_checkpoint["beta_std"]
)

Best epoch: 20
Best validation Macro-F1: 0.7343912095541211
Beta mean: 0.2837531864643097
Beta std: 0.026276707649230957


## Final test

In [ ]:
test_mg_results = evaluate_mlp_gcn(
    mlp_gcn_model,
    test_tab_tensor,
    test_graph
)

print(
    "Test Accuracy:",
    test_mg_results["accuracy"]
)

print(
    "Test Balanced Accuracy:",
    test_mg_results["balanced_accuracy"]
)

print(
    "Test Macro-F1:",
    test_mg_results["macro_f1"]
)

print(
    "Test MCC:",
    test_mg_results["mcc"]
)

print(
    "Beta Mean:",
    test_mg_results["beta_mean"]
)

print(
    "Beta Std:",
    test_mg_results["beta_std"]
)

Test Accuracy: 0.8204835529729169
Test Balanced Accuracy: 0.8262120157174097
Test Macro-F1: 0.7220104925149284
Test MCC: 0.6900858272078876
Beta Mean: 0.2828744053840637
Beta Std: 0.02542923018336296


## Classification report

In [ ]:
print(
    classification_report(
        test_mg_results["y_true"],
        test_mg_results["y_pred"],
        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.8747    0.8661    0.8704      3435
  Pathogenic     0.3061    0.8193    0.4457       476
         VUS     0.9154    0.7932    0.8499      5726

    accuracy                         0.8205      9637
   macro avg     0.6987    0.8262    0.7220      9637
weighted avg     0.8708    0.8205    0.8373      9637



# V2

## Restore both models

In [ ]:
# ============================================================
# RESTORE BEST BIOMARKER MODEL
# ============================================================

biomarker_checkpoint = torch.load(
    best_tab_path,
    map_location=device
)

biomarker_model.load_state_dict(
    biomarker_checkpoint[
        "model_state_dict"
    ]
)

biomarker_model.eval()


# ============================================================
# RESTORE BEST GRAPH V2 MODEL
# ============================================================

gcn_v2_checkpoint = torch.load(
    best_gcn_v2_path,
    map_location=device
)

gcn_model_v2.load_state_dict(
    gcn_v2_checkpoint[
        "model_state_dict"
    ]
)

gcn_model_v2.eval()


print("✅ Best Biomarker MLP restored")
print("✅ Best Graph v2 GCN restored")

✅ Best Biomarker MLP restored
✅ Best Graph v2 GCN restored


## Support functions

In [68]:
@torch.no_grad()
def get_biomarker_probabilities(
    model,
    X
):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    logits = model(
        X_tensor
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    return (
        probabilities
        .cpu()
        .numpy()
    )

In [69]:
@torch.no_grad()
def get_gcn_probabilities(
    model,
    graph
):

    model.eval()

    logits = model(
        graph.x,
        graph.edge_index
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    return (
        probabilities
        .cpu()
        .numpy()
    )

In [70]:
bio_val_prob = get_biomarker_probabilities(
    biomarker_model,
    X_val_tab
)

gcn_val_prob = get_gcn_probabilities(
    gcn_model_v2,
    val_graph_v2
)


y_val = (
    val_df[
        "Label"
    ]
    .to_numpy()
)


print(
    "Biomarker:",
    bio_val_prob.shape
)

print(
    "GCN:",
    gcn_val_prob.shape
)

print(
    "Labels:",
    y_val.shape
)


assert (
    bio_val_prob.shape
    ==
    gcn_val_prob.shape
)

assert (
    len(y_val)
    ==
    bio_val_prob.shape[0]
)


print(
    "✅ Validation predictions aligned"
)

Biomarker: (10298, 3)
GCN: (10298, 3)
Labels: (10298,)
✅ Validation predictions aligned


## Search best beta

In [ ]:
beta_results = []


for beta in np.arange(
    0.0,
    1.01,
    0.05
):

    fused_prob = (
        (1.0 - beta)
        *
        bio_val_prob
        +
        beta
        *
        gcn_val_prob
    )


    fused_pred = np.argmax(
        fused_prob,
        axis=1
    )


    macro_f1 = f1_score(
        y_val,
        fused_pred,
        average="macro",
        zero_division=0
    )


    balanced_acc = (
        balanced_accuracy_score(
            y_val,
            fused_pred
        )
    )


    accuracy = accuracy_score(
        y_val,
        fused_pred
    )


    mcc = matthews_corrcoef(
        y_val,
        fused_pred
    )


    beta_results.append({

        "beta":
            float(beta),

        "macro_f1":
            macro_f1,

        "balanced_accuracy":
            balanced_acc,

        "accuracy":
            accuracy,

        "mcc":
            mcc
    })


beta_results_df = pd.DataFrame(
    beta_results
)


display(
    beta_results_df
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

,beta,macro_f1,balanced_accuracy,accuracy,mcc
5,0.25,0.815456,0.831376,0.894543,0.795378
4,0.20,0.814176,0.831067,0.894057,0.794568
6,0.30,0.813904,0.831149,0.892892,0.792413
7,0.35,0.813440,0.831034,0.891921,0.790683
3,0.15,0.813062,0.830770,0.893669,0.793928
8,0.40,0.812769,0.832349,0.890755,0.788794
2,0.10,0.811514,0.829854,0.893086,0.792900
1,0.05,0.811008,0.830248,0.892795,0.792463
9,0.45,0.810157,0.830201,0.888328,0.784368
0,0.00,0.809384,0.828885,0.892406,0.791770


In [ ]:
best_beta_row = (
    beta_results_df
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .iloc[0]
)


BEST_BETA = float(
    best_beta_row[
        "beta"
    ]
)


print(
    "Best beta:",
    BEST_BETA
)

print(
    "Best validation Macro-F1:",
    best_beta_row[
        "macro_f1"
    ]
)

print(
    "Balanced Accuracy:",
    best_beta_row[
        "balanced_accuracy"
    ]
)

print(
    "Accuracy:",
    best_beta_row[
        "accuracy"
    ]
)

print(
    "MCC:",
    best_beta_row[
        "mcc"
    ]
)

Best beta: 0.25
Best validation Macro-F1: 0.8154557187368225
Balanced Accuracy: 0.8313758294379404
Accuracy: 0.8945426296368227
MCC: 0.7953775234747219


## Evaluate best beta and results

In [ ]:
bio_test_prob = get_biomarker_probabilities(
    biomarker_model,
    X_test_tab
)

gcn_test_prob = get_gcn_probabilities(
    gcn_model_v2,
    test_graph_v2
)


y_test = (
    test_df[
        "Label"
    ]
    .to_numpy()
)


final_test_prob = (

    (1.0 - BEST_BETA)
    *
    bio_test_prob

    +

    BEST_BETA
    *
    gcn_test_prob
)


final_test_pred = np.argmax(
    final_test_prob,
    axis=1
)

In [ ]:
late_fusion_accuracy = (
    accuracy_score(
        y_test,
        final_test_pred
    )
)


late_fusion_balanced = (
    balanced_accuracy_score(
        y_test,
        final_test_pred
    )
)


late_fusion_f1 = (
    f1_score(
        y_test,
        final_test_pred,
        average="macro",
        zero_division=0
    )
)


late_fusion_mcc = (
    matthews_corrcoef(
        y_test,
        final_test_pred
    )
)


print(
    "======= LATE MLP + GCN FUSION ======="
)

print(
    "Beta:",
    BEST_BETA
)

print(
    f"Accuracy: "
    f"{late_fusion_accuracy:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{late_fusion_balanced:.4f}"
)

print(
    f"Macro-F1: "
    f"{late_fusion_f1:.4f}"
)

print(
    f"MCC: "
    f"{late_fusion_mcc:.4f}"
)

======= LATE MLP + GCN FUSION =======
Beta: 0.25
Accuracy: 0.8958
Balanced Accuracy: 0.8331
Macro-F1: 0.8167
MCC: 0.7994


## Classification report

In [ ]:
print(
    classification_report(
        y_test,
        final_test_pred,
        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.9337    0.8611    0.8960      3435
  Pathogenic     0.5763    0.7059    0.6346       476
         VUS     0.9071    0.9324    0.9196      5726

    accuracy                         0.8958      9637
   macro avg     0.8057    0.8331    0.8167      9637
weighted avg     0.9002    0.8958    0.8971      9637



## TEMPORARY: Saving state

In [ ]:
# ============================================================
# SAVE FINAL SESSION STATE
# ============================================================

session_state = {
    "best_beta": float(BEST_BETA),

    "biomarker_test_accuracy":
        float(test_tab_results["accuracy"]),

    "biomarker_test_balanced_accuracy":
        float(test_tab_results["balanced_accuracy"]),

    "biomarker_test_macro_f1":
        float(test_tab_results["macro_f1"]),

    "graph_v2_test_accuracy":
        float(test_gcn_v2["accuracy"]),

    "graph_v2_test_balanced_accuracy":
        float(test_gcn_v2["balanced_accuracy"]),

    "graph_v2_test_macro_f1":
        float(test_gcn_v2["macro_f1"]),

    "late_fusion_accuracy":
        float(late_fusion_accuracy),

    "late_fusion_balanced_accuracy":
        float(late_fusion_balanced),

    "late_fusion_macro_f1":
        float(late_fusion_f1),

    "late_fusion_mcc":
        float(late_fusion_mcc),

    "seed":
        int(SEED)
}


torch.save(
    session_state,
    CHECKPOINT_DIR /
    "late_fusion_session_state.pt"
)


print("✅ Session state saved")
print(session_state)

✅ Session state saved
{'best_beta': 0.25, 'biomarker_test_accuracy': 0.8942616996990764, 'biomarker_test_balanced_accuracy': 0.8345282031127793, 'biomarker_test_macro_f1': 0.8135369026242429, 'graph_v2_test_accuracy': 0.7905987340458649, 'graph_v2_test_balanced_accuracy': 0.720568687075589, 'graph_v2_test_macro_f1': 0.6811191827519218, 'late_fusion_accuracy': 0.8958182006848604, 'late_fusion_balanced_accuracy': 0.8331437587793896, 'late_fusion_macro_f1': 0.8166944187398207, 'late_fusion_mcc': 0.7993674571291108, 'seed': 42}


In [ ]:
beta_results_df.to_csv(
    RESULTS_DIR /
    "late_fusion_beta_search.csv",
    index=False
)

print("✅ Beta search results saved")

✅ Beta search results saved


In [ ]:
final_predictions_df = pd.DataFrame({
    "true_label": y_test,
    "predicted_label": final_test_pred,

    "prob_benign":
        final_test_prob[:, 0],

    "prob_pathogenic":
        final_test_prob[:, 1],

    "prob_vus":
        final_test_prob[:, 2]
})


final_predictions_df.to_csv(
    RESULTS_DIR /
    "late_fusion_test_predictions.csv",
    index=False
)

print("✅ Final predictions saved")

✅ Final predictions saved


## RESTORE POINT

In [10]:
session_state = torch.load(
    CHECKPOINT_DIR /
    "late_fusion_session_state.pt",
    map_location="cpu"
)

BEST_BETA = session_state[
    "best_beta"
]

print("Restored beta:", BEST_BETA)

print(
    "Previous best Macro-F1:",
    session_state[
        "late_fusion_macro_f1"
    ]
)

Restored beta: 0.25
Previous best Macro-F1: 0.8166944187398207


In [23]:
from pathlib import Path

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/variant_effect_prediction/checkpoints"
)

RESULTS_DIR = Path(
    "/content/drive/MyDrive/variant_effect_prediction/results"
)

best_tab_path = (
    CHECKPOINT_DIR /
    "biomarker_mlp_best.pt"
)

best_gcn_v2_path = (
    CHECKPOINT_DIR /
    "gcn_v2_best.pt"
)

best_model_path = (
    CHECKPOINT_DIR /
    "sequence_cnn_best.pt"
)

print("Biomarker checkpoint:", best_tab_path)
print("GCN v2 checkpoint:", best_gcn_v2_path)
print("Sequence checkpoint:", best_model_path)

Biomarker checkpoint: /content/drive/MyDrive/variant_effect_prediction/checkpoints/biomarker_mlp_best.pt
GCN v2 checkpoint: /content/drive/MyDrive/variant_effect_prediction/checkpoints/gcn_v2_best.pt
Sequence checkpoint: /content/drive/MyDrive/variant_effect_prediction/checkpoints/sequence_cnn_best.pt


In [37]:
biomarker_checkpoint = torch.load(
    best_tab_path,
    map_location=device
)

biomarker_model.load_state_dict(
    biomarker_checkpoint[
        "model_state_dict"
    ]
)

biomarker_model.eval()

print("✅ Biomarker model restored")

✅ Biomarker model restored


In [64]:
gcn_v2_checkpoint = torch.load(
    best_gcn_v2_path,
    map_location=device
)

gcn_model_v2.load_state_dict(
    gcn_v2_checkpoint[
        "model_state_dict"
    ]
)

gcn_model_v2.eval()

print("✅ Graph v2 restored")

✅ Graph v2 restored


In [65]:
sequence_checkpoint = torch.load(
    best_model_path,
    map_location=device
)

sequence_model.load_state_dict(
    sequence_checkpoint[
        "model_state_dict"
    ]
)

sequence_model.eval()

print("✅ Sequence CNN restored")

✅ Sequence CNN restored


### Getting validation predictions

In [66]:
@torch.no_grad()
def get_sequence_probabilities(
    model,
    loader
):

    model.eval()

    all_probabilities = []
    all_labels = []

    for seq, targets in loader:

        seq = seq.to(device)

        logits = model(
            seq
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        all_probabilities.append(
            probabilities
            .cpu()
            .numpy()
        )

        all_labels.append(
            targets.numpy()
        )

    probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    labels = np.concatenate(
        all_labels,
        axis=0
    )

    return probabilities, labels

In [72]:
# Biomarker
bio_val_prob = get_biomarker_probabilities(
    biomarker_model,
    X_val_tab
)


# Graph v2
gcn_val_prob = get_gcn_probabilities(
    gcn_model_v2,
    val_graph_v2
)


# Sequence
seq_val_prob, seq_val_labels = (
    get_sequence_probabilities(
        sequence_model,
        val_loader
    )
)


y_val = (
    val_df[
        "Label"
    ]
    .to_numpy()
)

In [73]:
print(
    "Bio:",
    bio_val_prob.shape
)

print(
    "Graph:",
    gcn_val_prob.shape
)

print(
    "Sequence:",
    seq_val_prob.shape
)

print(
    "Labels:",
    y_val.shape
)


assert bio_val_prob.shape == gcn_val_prob.shape
assert bio_val_prob.shape == seq_val_prob.shape

assert np.array_equal(
    y_val,
    seq_val_labels
)

print(
    "✅ All three modalities aligned"
)

Bio: (10298, 3)
Graph: (10298, 3)
Sequence: (10298, 3)
Labels: (10298,)
✅ All three modalities aligned


## 3 way weight search

In [74]:
tri_results = []


for w_seq in np.arange(
    0.0,
    0.31,
    0.05
):

    for w_graph in np.arange(
        0.0,
        0.51,
        0.05
    ):

        w_bio = (
            1.0
            -
            w_seq
            -
            w_graph
        )


        if w_bio < 0:
            continue


        fused_prob = (
            w_bio
            *
            bio_val_prob

            +

            w_graph
            *
            gcn_val_prob

            +

            w_seq
            *
            seq_val_prob
        )


        predictions = np.argmax(
            fused_prob,
            axis=1
        )


        macro_f1 = f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0
        )


        balanced_acc = (
            balanced_accuracy_score(
                y_val,
                predictions
            )
        )


        accuracy = accuracy_score(
            y_val,
            predictions
        )


        mcc = matthews_corrcoef(
            y_val,
            predictions
        )


        tri_results.append({

            "w_bio":
                float(w_bio),

            "w_graph":
                float(w_graph),

            "w_seq":
                float(w_seq),

            "macro_f1":
                macro_f1,

            "balanced_accuracy":
                balanced_acc,

            "accuracy":
                accuracy,

            "mcc":
                mcc
        })


tri_results_df = pd.DataFrame(
    tri_results
)

In [75]:
display(
    tri_results_df
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .head(15)
)

,w_bio,w_graph,w_seq,macro_f1,balanced_accuracy,accuracy,mcc
72,0.40,0.30,0.30,0.819496,0.830557,0.893086,0.792211
71,0.45,0.25,0.30,0.818292,0.830255,0.894154,0.794269
70,0.50,0.20,0.30,0.817890,0.829399,0.895222,0.796202
59,0.55,0.20,0.25,0.817855,0.829903,0.895514,0.796794
61,0.45,0.30,0.25,0.817720,0.830241,0.892892,0.791965
48,0.60,0.20,0.20,0.817588,0.830469,0.895611,0.797072
51,0.45,0.35,0.20,0.817479,0.831716,0.892018,0.790584
60,0.50,0.25,0.25,0.817365,0.829618,0.894154,0.794288
49,0.55,0.25,0.20,0.817339,0.830718,0.894446,0.794959
40,0.50,0.35,0.15,0.817264,0.832894,0.891824,0.790303


In [76]:
best_tri_row = (
    tri_results_df
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .iloc[0]
)


W_BIO = float(
    best_tri_row[
        "w_bio"
    ]
)

W_GRAPH = float(
    best_tri_row[
        "w_graph"
    ]
)

W_SEQ = float(
    best_tri_row[
        "w_seq"
    ]
)


print(
    "===== BEST TRI-MODAL WEIGHTS ====="
)

print(
    "Biomarker:",
    W_BIO
)

print(
    "Graph:",
    W_GRAPH
)

print(
    "Sequence:",
    W_SEQ
)

print(
    "\nValidation Macro-F1:",
    best_tri_row[
        "macro_f1"
    ]
)

print(
    "Validation Balanced Accuracy:",
    best_tri_row[
        "balanced_accuracy"
    ]
)

print(
    "Validation Accuracy:",
    best_tri_row[
        "accuracy"
    ]
)

print(
    "Validation MCC:",
    best_tri_row[
        "mcc"
    ]
)

===== BEST TRI-MODAL WEIGHTS =====
Biomarker: 0.3999999999999999
Graph: 0.30000000000000004
Sequence: 0.30000000000000004

Validation Macro-F1: 0.8194955689341539
Validation Balanced Accuracy: 0.8305573579776927
Validation Accuracy: 0.8930860361235191
Validation MCC: 0.7922105296721434


### Testing

In [77]:
bio_test_prob = get_biomarker_probabilities(
    biomarker_model,
    X_test_tab
)


gcn_test_prob = get_gcn_probabilities(
    gcn_model_v2,
    test_graph_v2
)


seq_test_prob, seq_test_labels = (
    get_sequence_probabilities(
        sequence_model,
        test_loader
    )
)


y_test = (
    test_df[
        "Label"
    ]
    .to_numpy()
)


assert np.array_equal(
    y_test,
    seq_test_labels
)

In [78]:
tri_test_prob = (

    W_BIO
    *
    bio_test_prob

    +

    W_GRAPH
    *
    gcn_test_prob

    +

    W_SEQ
    *
    seq_test_prob
)

In [79]:
tri_test_pred = np.argmax(
    tri_test_prob,
    axis=1
)

In [80]:
tri_accuracy = accuracy_score(
    y_test,
    tri_test_pred
)


tri_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        tri_test_pred
    )
)


tri_macro_f1 = f1_score(
    y_test,
    tri_test_pred,
    average="macro",
    zero_division=0
)


tri_mcc = matthews_corrcoef(
    y_test,
    tri_test_pred
)


print(
    "========= TRI-MODAL LATE FUSION ========="
)

print(
    f"Accuracy: "
    f"{tri_accuracy:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{tri_balanced_accuracy:.4f}"
)

print(
    f"Macro-F1: "
    f"{tri_macro_f1:.4f}"
)

print(
    f"MCC: "
    f"{tri_mcc:.4f}"
)


print(
    "\nWeights:"
)

print(
    f"Biomarker = {W_BIO:.2f}"
)

print(
    f"Graph     = {W_GRAPH:.2f}"
)

print(
    f"Sequence  = {W_SEQ:.2f}"
)

========= TRI-MODAL LATE FUSION =========
Accuracy: 0.8913
Balanced Accuracy: 0.8255
Macro-F1: 0.8145
MCC: 0.7901

Weights:
Biomarker = 0.40
Graph     = 0.30
Sequence  = 0.30


In [81]:
print(
    classification_report(
        y_test,
        tri_test_pred,
        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.9168    0.8664    0.8909      3435
  Pathogenic     0.5967    0.6870    0.6387       476
         VUS     0.9047    0.9232    0.9138      5726

    accuracy                         0.8913      9637
   macro avg     0.8061    0.8255    0.8145      9637
weighted avg     0.8938    0.8913    0.8921      9637



# Trimodal gated network

## Model

In [82]:
class TriModalGate(nn.Module):

    def __init__(self):

        super().__init__()

        # 3 probabilities from each model
        # = 9 total inputs

        self.gate_network = nn.Sequential(

            nn.Linear(9, 32),

            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(32, 16),

            nn.ReLU(),

            nn.Linear(16, 3)
        )


    def forward(
        self,
        bio_prob,
        graph_prob,
        seq_prob
    ):

        gate_input = torch.cat(
            [
                bio_prob,
                graph_prob,
                seq_prob
            ],
            dim=1
        )


        gate_logits = self.gate_network(
            gate_input
        )


        weights = torch.softmax(
            gate_logits,
            dim=1
        )


        w_bio = weights[:, 0:1]
        w_graph = weights[:, 1:2]
        w_seq = weights[:, 2:3]


        final_prob = (
            w_bio * bio_prob
            +
            w_graph * graph_prob
            +
            w_seq * seq_prob
        )


        return (
            final_prob,
            weights
        )

In [83]:
class TriModalGate(nn.Module):

    def __init__(self):

        super().__init__()

        self.hidden = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.output = nn.Linear(
            16,
            3
        )

        nn.init.zeros_(
            self.output.weight
        )

        nn.init.constant_(
            self.output.bias,
            0.0
        )

        with torch.no_grad():

            self.output.bias[:] = torch.tensor([
                1.5,   # biomarker
                0.3,   # graph
               -0.4    # sequence
            ])


    def forward(
        self,
        bio_prob,
        graph_prob,
        seq_prob
    ):

        gate_input = torch.cat(
            [
                bio_prob,
                graph_prob,
                seq_prob
            ],
            dim=1
        )

        hidden = self.hidden(
            gate_input
        )

        gate_logits = self.output(
            hidden
        )

        weights = torch.softmax(
            gate_logits,
            dim=1
        )

        final_prob = (
            weights[:, 0:1] * bio_prob
            +
            weights[:, 1:2] * graph_prob
            +
            weights[:, 2:3] * seq_prob
        )

        return (
            final_prob,
            weights
        )

In [87]:
gate_model = TriModalGate().to(device)

print(gate_model)

TriModalGate(
  (hidden): Sequential(
    (0): Linear(in_features=9, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
  )
  (output): Linear(in_features=16, out_features=3, bias=True)
)


In [88]:
bio_sample = torch.tensor(
    bio_val_prob[:256],
    dtype=torch.float32,
    device=device
)

gcn_sample = torch.tensor(
    gcn_val_prob[:256],
    dtype=torch.float32,
    device=device
)

seq_sample = torch.tensor(
    seq_val_prob[:256],
    dtype=torch.float32,
    device=device
)


gate_model.eval()

with torch.no_grad():

    final_prob, gate_weights = gate_model(
        bio_sample,
        gcn_sample,
        seq_sample
    )


print(
    "Mean initial gate weights:",
    gate_weights.mean(dim=0)
)

Mean initial gate weights: tensor([0.6893, 0.2076, 0.1031])


## Dataset creation

In [89]:
class GateDataset(Dataset):

    def __init__(
        self,
        bio_prob,
        graph_prob,
        seq_prob,
        labels
    ):

        self.bio = torch.tensor(
            bio_prob,
            dtype=torch.float32
        )

        self.graph = torch.tensor(
            graph_prob,
            dtype=torch.float32
        )

        self.seq = torch.tensor(
            seq_prob,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )


    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):

        return (
            self.bio[idx],
            self.graph[idx],
            self.seq[idx],
            self.labels[idx]
        )

## Getting probabilities

In [91]:
y_train = train_df["Label"].to_numpy()
y_val   = val_df["Label"].to_numpy()
y_test  = test_df["Label"].to_numpy()

In [92]:
bio_train_prob = get_biomarker_probabilities(
    biomarker_model,
    X_train_tab
)

bio_val_prob = get_biomarker_probabilities(
    biomarker_model,
    X_val_tab
)

bio_test_prob = get_biomarker_probabilities(
    biomarker_model,
    X_test_tab
)

In [93]:
gcn_train_prob = get_gcn_probabilities(
    gcn_model_v2,
    train_graph_v2
)

gcn_val_prob = get_gcn_probabilities(
    gcn_model_v2,
    val_graph_v2
)

gcn_test_prob = get_gcn_probabilities(
    gcn_model_v2,
    test_graph_v2
)

In [94]:
seq_val_prob, seq_val_labels = (
    get_sequence_probabilities(
        sequence_model,
        val_loader
    )
)

seq_test_prob, seq_test_labels = (
    get_sequence_probabilities(
        sequence_model,
        test_loader
    )
)

In [95]:
train_eval_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [96]:
seq_train_prob, seq_train_labels = (
    get_sequence_probabilities(
        sequence_model,
        train_eval_loader
    )
)

In [97]:
print("TRAIN")
print(
    bio_train_prob.shape,
    gcn_train_prob.shape,
    seq_train_prob.shape,
    y_train.shape
)

print("\nVAL")
print(
    bio_val_prob.shape,
    gcn_val_prob.shape,
    seq_val_prob.shape,
    y_val.shape
)

print("\nTEST")
print(
    bio_test_prob.shape,
    gcn_test_prob.shape,
    seq_test_prob.shape,
    y_test.shape
)

TRAIN
(80065, 3) (80065, 3) (80065, 3) (80065,)

VAL
(10298, 3) (10298, 3) (10298, 3) (10298,)

TEST
(9637, 3) (9637, 3) (9637, 3) (9637,)


In [98]:
gate_train_dataset = GateDataset(
    bio_train_prob,
    gcn_train_prob,
    seq_train_prob,
    y_train
)

gate_val_dataset = GateDataset(
    bio_val_prob,
    gcn_val_prob,
    seq_val_prob,
    y_val
)

gate_test_dataset = GateDataset(
    bio_test_prob,
    gcn_test_prob,
    seq_test_prob,
    y_test
)

## Dataloaders

In [100]:
gate_train_loader = DataLoader(
    gate_train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=0
)

gate_val_loader = DataLoader(
    gate_val_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

gate_test_loader = DataLoader(
    gate_test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

In [101]:
gate_optimizer = torch.optim.AdamW(
    gate_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

In [102]:
prior = torch.tensor(
    [0.70, 0.20, 0.10],
    dtype=torch.float32,
    device=device
)

## Helpers

In [104]:
def compute_gate_loss(
    final_prob,
    gate_weights,
    targets
):

    final_prob = torch.clamp(
        final_prob,
        min=1e-7,
        max=1.0
    )

    log_prob = torch.log(
        final_prob
    )

    classification_loss = F.nll_loss(
        log_prob,
        targets,
        weight=class_weights
    )


    mean_gate_weights = (
        gate_weights.mean(dim=0)
    )


    regularization = F.mse_loss(
        mean_gate_weights,
        prior
    )


    total_loss = (
        classification_loss
        +
        0.05 * regularization
    )


    return total_loss

In [105]:
@torch.no_grad()
def evaluate_gate(
    model,
    loader
):

    model.eval()

    all_true = []
    all_pred = []
    all_prob = []
    all_gate_weights = []


    for (
        bio_prob,
        graph_prob,
        seq_prob,
        targets
    ) in loader:


        bio_prob = bio_prob.to(device)

        graph_prob = graph_prob.to(device)

        seq_prob = seq_prob.to(device)

        targets = targets.to(device)


        final_prob, gate_weights = model(
            bio_prob,
            graph_prob,
            seq_prob
        )


        predictions = torch.argmax(
            final_prob,
            dim=1
        )


        all_true.extend(
            targets.cpu().numpy()
        )

        all_pred.extend(
            predictions.cpu().numpy()
        )

        all_prob.extend(
            final_prob.cpu().numpy()
        )

        all_gate_weights.extend(
            gate_weights.cpu().numpy()
        )


    y_true = np.asarray(
        all_true
    )

    y_pred = np.asarray(
        all_pred
    )

    y_prob = np.asarray(
        all_prob
    )

    gate_weights = np.asarray(
        all_gate_weights
    )


    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "mean_weights":
            gate_weights.mean(
                axis=0
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob,

        "gate_weights":
            gate_weights
    }

## Training

In [107]:
GATE_EPOCHS = 20
GATE_PATIENCE = 4

best_gate_f1 = -1.0
no_improvement = 0

best_gate_path = (
    CHECKPOINT_DIR /
    "trimodal_gate_best.pt"
)

In [108]:
for epoch in range(
    GATE_EPOCHS
):

    gate_model.train()

    running_loss = 0.0


    for (
        bio_prob,
        graph_prob,
        seq_prob,
        targets
    ) in gate_train_loader:


        bio_prob = bio_prob.to(device)

        graph_prob = graph_prob.to(device)

        seq_prob = seq_prob.to(device)

        targets = targets.to(device)


        gate_optimizer.zero_grad(
            set_to_none=True
        )


        final_prob, gate_weights = gate_model(
            bio_prob,
            graph_prob,
            seq_prob
        )


        loss = compute_gate_loss(
            final_prob,
            gate_weights,
            targets
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            gate_model.parameters(),
            5.0
        )


        gate_optimizer.step()


        running_loss += (
            loss.item()
            *
            targets.size(0)
        )


    train_loss = (
        running_loss
        /
        len(gate_train_dataset)
    )


    val_results = evaluate_gate(
        gate_model,
        gate_val_loader
    )


    val_f1 = (
        val_results[
            "macro_f1"
        ]
    )


    mean_weights = (
        val_results[
            "mean_weights"
        ]
    )


    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "
        f"Val Bal Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "
        f"Val Macro-F1: "
        f"{val_f1:.4f} | "
        f"Val MCC: "
        f"{val_results['mcc']:.4f}"
    )


    print(
        "Mean gate weights | "
        f"Bio: {mean_weights[0]:.3f} | "
        f"Graph: {mean_weights[1]:.3f} | "
        f"Seq: {mean_weights[2]:.3f}"
    )


    if val_f1 > best_gate_f1 + 1e-4:

        best_gate_f1 = val_f1

        no_improvement = 0


        torch.save(
            {
                "model_state_dict":
                    gate_model.state_dict(),

                "epoch":
                    epoch + 1,

                "val_macro_f1":
                    best_gate_f1,

                "mean_weights":
                    mean_weights
            },

            best_gate_path
        )


        print(
            "✓ Best gate saved."
        )


    else:

        no_improvement += 1

        print(
            f"No improvement: "
            f"{no_improvement}/"
            f"{GATE_PATIENCE}"
        )


    if no_improvement >= GATE_PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break

Epoch 01 | Loss: 0.4542 | Val Acc: 0.8928 | Val Bal Acc: 0.8302 | Val Macro-F1: 0.8109 | Val MCC: 0.7925
Mean gate weights | Bio: 0.955 | Graph: 0.031 | Seq: 0.014
✓ Best gate saved.
Epoch 02 | Loss: 0.4341 | Val Acc: 0.8928 | Val Bal Acc: 0.8302 | Val Macro-F1: 0.8109 | Val MCC: 0.7925
Mean gate weights | Bio: 0.972 | Graph: 0.022 | Seq: 0.007
No improvement: 1/4
Epoch 03 | Loss: 0.4334 | Val Acc: 0.8928 | Val Bal Acc: 0.8302 | Val Macro-F1: 0.8109 | Val MCC: 0.7925
Mean gate weights | Bio: 0.959 | Graph: 0.033 | Seq: 0.008
No improvement: 2/4
Epoch 04 | Loss: 0.4328 | Val Acc: 0.8927 | Val Bal Acc: 0.8302 | Val Macro-F1: 0.8107 | Val MCC: 0.7923
Mean gate weights | Bio: 0.946 | Graph: 0.046 | Seq: 0.009
No improvement: 3/4
Epoch 05 | Loss: 0.4313 | Val Acc: 0.8928 | Val Bal Acc: 0.8302 | Val Macro-F1: 0.8109 | Val MCC: 0.7925
Mean gate weights | Bio: 0.932 | Graph: 0.058 | Seq: 0.010
No improvement: 4/4

Early stopping triggered.


## Results

In [111]:
gate_checkpoint = torch.load(
    best_gate_path,
    map_location=device,
    weights_only=False
)

gate_model.load_state_dict(
    gate_checkpoint[
        "model_state_dict"
    ]
)

print(
    "Best epoch:",
    gate_checkpoint["epoch"]
)

print(
    "Best validation Macro-F1:",
    gate_checkpoint["val_macro_f1"]
)

print(
    "Mean gate weights:",
    gate_checkpoint["mean_weights"]
)

Best epoch: 1
Best validation Macro-F1: 0.810861458589593
Mean gate weights: [0.95484304 0.03146965 0.01368629]


In [112]:
gate_test_results = evaluate_gate(
    gate_model,
    gate_test_loader
)

print(
    "========= GATED TRI-MODAL ========="
)

print(
    f"Accuracy: "
    f"{gate_test_results['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{gate_test_results['balanced_accuracy']:.4f}"
)

print(
    f"Macro-F1: "
    f"{gate_test_results['macro_f1']:.4f}"
)

print(
    f"MCC: "
    f"{gate_test_results['mcc']:.4f}"
)

print(
    "\nAverage modality weights:"
)

print(
    f"Biomarker: "
    f"{gate_test_results['mean_weights'][0]:.3f}"
)

print(
    f"Graph: "
    f"{gate_test_results['mean_weights'][1]:.3f}"
)

print(
    f"Sequence: "
    f"{gate_test_results['mean_weights'][2]:.3f}"
)

========= GATED TRI-MODAL =========
Accuracy: 0.8944
Balanced Accuracy: 0.8346
Macro-F1: 0.8138
MCC: 0.7971

Average modality weights:
Biomarker: 0.955
Graph: 0.031
Sequence: 0.014


## Classification report

In [117]:
print(
    classification_report(
        gate_test_results[
            "y_true"
        ],
        gate_test_results[
            "y_pred"
        ],
        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.9344    0.8588    0.8950      3435
  Pathogenic     0.5592    0.7143    0.6273       476
         VUS     0.9075    0.9307    0.9190      5726

    accuracy                         0.8944      9637
   macro avg     0.8004    0.8346    0.8138      9637
weighted avg     0.8999    0.8944    0.8960      9637



In [119]:
print("Biomarker:", W_BIO)
print("Graph:", W_GRAPH)
print("Sequence:", W_SEQ)

Biomarker: 0.3999999999999999
Graph: 0.30000000000000004
Sequence: 0.30000000000000004


### Closing

In [121]:
final_handoff = {
    "architecture": "Tri-modal weighted late fusion",
    "weights": {
        "biomarker": 0.40,
        "graph": 0.30,
        "sequence": 0.30
    },

    "test_accuracy": 0.8913,
    "test_balanced_accuracy": 0.8255,
    "test_macro_f1": 0.8145,

    "benign_f1": 0.8909,
    "pathogenic_f1": 0.6387,
    "vus_f1": 0.9138,

    "graph_version": "v2",
    "sequence_window": 101,

    "graph_parameters": {
        "k_same_consequence": 4,
        "k_conservation": 3,
        "k_prox": 2,
        "max_bp": 5000
    },

    "seed": SEED
}

torch.save(
    final_handoff,
    CHECKPOINT_DIR /
    "genohybrid_locked_architecture.pt"
)

print("✅ Locked architecture saved")

✅ Locked architecture saved
